# 00 · Cài đặt thư viện & kiểm thử

Notebook này chạy **một lần** (hoặc mỗi khi có bản cập nhật thư viện). Nó:
1. Kết nối Google Drive, tạo thư mục đồ án `MyDrive/NLP_Final_EnVi/`.
2. Ghi thư viện dùng chung `envi_lib/` (mô hình, huấn luyện, giải mã, độ đo) lên Drive — mọi notebook sau đều dùng nó,
   nên cách xử lý dữ liệu và chấm điểm là **thống nhất**.
3. Chạy **bộ kiểm thử tự động** (13 bài) để chắc chắn mọi thứ đúng trước khi tốn giờ GPU.

**Thứ tự chạy toàn bộ đồ án:** `00` → `01` → `02` → `03` → `04` → `05` → `06`.

Phiên bản thư viện: **1.0.0**

In [2]:
#@title 🔧 Kết nối Drive, tạo thư mục đồ án
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
PROJECT_DIR = os.environ.get("ENVI_PROJECT_DIR", "/content/drive/MyDrive/NLP_Final_EnVi")
os.makedirs(os.path.join(PROJECT_DIR, "envi_lib"), exist_ok=True)
os.chdir(PROJECT_DIR)
print("Thư mục đồ án:", os.getcwd())

Mounted at /content/drive
Thư mục đồ án: /content/drive/MyDrive/NLP_Final_EnVi


## Ghi thư viện `envi_lib` lên Drive
Mỗi ô dưới đây ghi một file. Có thể mở các file này trên Drive để đọc mã nguồn (dùng cho phần trình bày trong báo cáo).

In [3]:
%%writefile envi_lib/__init__.py
"""envi_lib — thư viện dùng chung cho đồ án dịch máy Anh-Việt bằng Transformer."""
from .common import LIB_VERSION  # noqa: F401

Overwriting envi_lib/__init__.py


In [4]:
%%writefile envi_lib/common.py
"""Tiện ích dùng chung cho toàn bộ đồ án dịch máy Anh -> Việt.

Mọi notebook đều import module này để dùng chung: đường dẫn, seed, chuẩn hoá văn bản,
chọn kiểu số thực (AMP), lưu/đọc file dự đoán.  Giữ một nguồn duy nhất để tránh mỗi
notebook xử lý một kiểu dẫn đến số liệu không so sánh được.
"""
from __future__ import annotations

import contextlib
import datetime
import json
import os
import random
import re
import shutil
import time
import unicodedata
from pathlib import Path

import numpy as np
import torch

LIB_VERSION = "1.0.0"

# ----------------------------------------------------------------------------
# Môi trường & đường dẫn
# ----------------------------------------------------------------------------

def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/NLP_Final_EnVi"


class Paths:
    """Cấu trúc thư mục của đồ án.

    root/
      data_raw/       dữ liệu gốc tải về (dùng chung cho DRY_RUN và chạy thật)
      data_clean/     dữ liệu đã làm sạch (dùng chung)
      spm/            tokenizer SentencePiece
      checkpoints/    checkpoint huấn luyện
      predictions/    bản dịch tập test của từng hệ thống (*.vi.txt + *.meta.json)
      results/        bảng số liệu, hình vẽ
    Khi dry_run=True, các thư mục spm/checkpoints/predictions/results nằm trong root/dryrun/
    để kết quả chạy thử không lẫn vào kết quả thật.
    """

    def __init__(self, root: str | os.PathLike, dry_run: bool = False):
        self.root = Path(root)
        self.dry_run = dry_run
        work = self.root / "dryrun" if dry_run else self.root
        self.work = work
        self.data_raw = self.root / "data_raw"
        self.data_clean = self.root / "data_clean"
        self.spm = work / "spm"
        self.checkpoints = work / "checkpoints"
        self.predictions = work / "predictions"
        self.results = work / "results"
        self.figures = self.results / "figures"

    def make_all(self):
        for p in [self.data_raw, self.data_clean, self.spm, self.checkpoints,
                  self.predictions, self.results, self.figures]:
            p.mkdir(parents=True, exist_ok=True)
        return self

    @property
    def spm_model(self) -> Path:
        return self.spm / "spm_envi.model"

    def __repr__(self):
        return f"Paths(root={self.root}, dry_run={self.dry_run})"


def get_project_root(default: str = DEFAULT_DRIVE_DIR) -> Path:
    """Thư mục gốc của đồ án.  Có thể ghi đè bằng biến môi trường ENVI_PROJECT_DIR."""
    env = os.environ.get("ENVI_PROJECT_DIR")
    return Path(env) if env else Path(default)


# ----------------------------------------------------------------------------
# Seed, thiết bị, AMP
# ----------------------------------------------------------------------------

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def bf16_supported() -> bool:
    return torch.cuda.is_available() and torch.cuda.is_bf16_supported()


def pick_amp_dtype(prefer: str = "auto"):
    """Chọn kiểu số thực cho autocast.

    - GPU hỗ trợ bf16 (A100, L4, H100): bf16 — ổn định, không cần GradScaler.
    - GPU cũ (T4, V100): fp16 + GradScaler.
    - CPU: None (chạy fp32).
    prefer: "auto" | "bf16" | "fp16" | "fp32"
    """
    if not torch.cuda.is_available() or prefer == "fp32":
        return None
    if prefer == "fp16":
        return torch.float16
    if prefer == "bf16":
        if not bf16_supported():
            raise RuntimeError("GPU này không hỗ trợ bf16. Dùng prefer='auto' hoặc 'fp16'.")
        return torch.bfloat16
    return torch.bfloat16 if bf16_supported() else torch.float16


def autocast_ctx(amp_dtype):
    if amp_dtype is None:
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda", dtype=amp_dtype)


def gpu_info() -> dict:
    info = {"cuda": torch.cuda.is_available(), "torch": torch.__version__}
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(name=p.name, total_gb=round(p.total_memory / 1024**3, 1),
                    bf16=torch.cuda.is_bf16_supported())
    return info


def peak_gpu_mem_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return round(torch.cuda.max_memory_allocated() / 1024**3, 2)


def reset_peak_mem():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def free_memory():
    """Giải phóng bộ nhớ GPU giữa các mô hình.  Gọi SAU khi đã `del model, optimizer, ...`."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def count_params(model, trainable_only: bool = False) -> int:
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


def disk_free_gb(path) -> float:
    path = Path(path)
    while not path.exists() and path != path.parent:
        path = path.parent
    return round(shutil.disk_usage(path).free / 1024**3, 1)


# ----------------------------------------------------------------------------
# Chuẩn hoá văn bản
# ----------------------------------------------------------------------------

_ZERO_WIDTH = re.compile(r"[​‌‍⁠﻿]")
_SPACES = re.compile(r"\s+")


def normalize_text(s: str) -> str:
    """Chuẩn hoá nhẹ, áp dụng thống nhất cho nguồn, tham chiếu và bản dịch.

    - Unicode NFC: tiếng Việt có thể ở dạng dựng sẵn hoặc tổ hợp (ví dụ 'ệ' = 1 hay 3 ký tự).
      Hai dạng nhìn giống nhau nhưng BLEU coi là khác nhau => bắt buộc chuẩn hoá.
    - Bỏ ký tự độ rộng 0, gộp khoảng trắng (kể cả xuống dòng) thành 1 dấu cách.
    Không đổi chữ hoa/thường, không đổi dấu câu (để không làm sai lệch tham chiếu).
    """
    if s is None:
        return ""
    s = unicodedata.normalize("NFC", str(s))
    s = _ZERO_WIDTH.sub("", s)
    s = _SPACES.sub(" ", s)
    return s.strip()


# ----------------------------------------------------------------------------
# Đọc / ghi file
# ----------------------------------------------------------------------------

def read_lines(path) -> list[str]:
    with open(path, encoding="utf-8") as f:
        return [line.rstrip("\n").rstrip("\r") for line in f]


def write_lines(path, lines):
    """Ghi mỗi phần tử một dòng.  Ghi ra file tạm rồi đổi tên (an toàn khi bị ngắt giữa chừng)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for line in lines:
            line = "" if line is None else str(line)
            if "\n" in line or "\r" in line:
                line = _SPACES.sub(" ", line)
            f.write(line + "\n")
    os.replace(tmp, path)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def now_str() -> str:
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


class Timer:
    def __init__(self):
        self.t0 = time.time()

    def elapsed(self) -> float:
        return time.time() - self.t0

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.seconds = self.elapsed()


def fmt_seconds(s: float) -> str:
    s = int(s)
    h, r = divmod(s, 3600)
    m, s = divmod(r, 60)
    return f"{h}h{m:02d}m{s:02d}s" if h else f"{m}m{s:02d}s"


# ----------------------------------------------------------------------------
# Lưu bản dịch của từng hệ thống
# ----------------------------------------------------------------------------

SYSTEM_NAME_RE = re.compile(r"^[A-Za-z0-9_\-\.]+$")


def save_predictions(paths: Paths, system: str, hyps: list[str], meta: dict | None = None,
                     expected_n: int | None = None):
    """Lưu bản dịch tập test: predictions/<system>.vi.txt và <system>.meta.json.

    Kiểm tra: số dòng phải bằng số câu test; cảnh báo nếu có nhiều câu rỗng.
    """
    if not SYSTEM_NAME_RE.match(system):
        raise ValueError(f"Tên hệ thống '{system}' chỉ được chứa chữ, số, '_', '-', '.'")
    hyps = [normalize_text(h) for h in hyps]
    if expected_n is not None and len(hyps) != expected_n:
        raise ValueError(f"[{system}] Có {len(hyps)} bản dịch nhưng tập test có {expected_n} câu.")
    n_empty = sum(1 for h in hyps if not h)
    if n_empty:
        print(f"⚠️  [{system}] {n_empty} bản dịch rỗng ({n_empty / max(1, len(hyps)):.1%}).")
    paths.predictions.mkdir(parents=True, exist_ok=True)
    write_lines(paths.predictions / f"{system}.vi.txt", hyps)
    meta = dict(meta or {})
    meta.update(system=system, n_lines=len(hyps), n_empty=n_empty, saved_at=now_str(),
                lib_version=LIB_VERSION, gpu=gpu_info().get("name", "cpu"))
    save_json(meta, paths.predictions / f"{system}.meta.json")
    print(f"✅ Đã lưu {len(hyps)} bản dịch: {paths.predictions / (system + '.vi.txt')}")


def check_lib_version(required: str):
    if LIB_VERSION != required:
        raise RuntimeError(
            f"Thư viện envi_lib trên Drive là bản {LIB_VERSION}, notebook này cần bản {required}. "
            "Hãy chạy lại notebook 00_cai_dat.ipynb để cập nhật thư viện.")


def prediction_is_current(paths: Paths, system: str, fingerprint: dict, expected_n: int) -> bool:
    """True nếu bản dịch của hệ thống đã tồn tại và được sinh từ đúng checkpoint/cấu hình hiện tại.

    Dùng để KHÔNG dịch lại tập test một cách thừa khi chạy lại notebook (tiết kiệm thời gian GPU).
    """
    mp = paths.predictions / f"{system}.meta.json"
    tp = paths.predictions / f"{system}.vi.txt"
    if not (mp.exists() and tp.exists()):
        return False
    try:
        meta = load_json(mp)
    except Exception:  # noqa: BLE001
        return False
    if meta.get("n_lines") != expected_n or meta.get("fingerprint") != fingerprint:
        return False
    print(f"✔ {system}: đã có bản dịch từ đúng checkpoint này — bỏ qua (xoá file trong predictions/ nếu muốn dịch lại).")
    return True

Overwriting envi_lib/common.py


In [5]:
%%writefile envi_lib/data.py
"""Tải, làm sạch, thống kê dữ liệu IWSLT'15 En-Vi; tokenizer SentencePiece; batching.

Nguồn dữ liệu (theo thứ tự ưu tiên):
  1. File đã có sẵn trong data_raw/stanford/ (người dùng có thể tự tải và đặt vào).
  2. Tải từ máy chủ Stanford NLP: train / tst2012 (dev) / tst2013 (test) — split chuẩn trong các bài báo.
  3. Dự phòng: bản parquet trên Hugging Face (IWSLT/mt_eng_vietnamese).
     LƯU Ý: bộ này có validation == test == tst2013, nên KHÔNG dùng validation của nó;
     tập dev được tách ngẫu nhiên 1.500 câu từ train (cố định seed).
"""
from __future__ import annotations

import html
import os
import random
import time
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch

from .common import Paths, load_json, normalize_text, read_lines, save_json, write_lines

STANFORD_BASE = "https://nlp.stanford.edu/projects/nmt/data/iwslt15.en-vi/"
STANFORD_SPLITS = {"train": "train", "dev": "tst2012", "test": "tst2013"}
HF_REPO = "IWSLT/mt_eng_vietnamese"
HF_REVISION = "refs/convert/parquet"
HF_FILES = {"train": "iwslt2015-en-vi/train/0000.parquet",
            "test": "iwslt2015-en-vi/test/0000.parquet"}
# Số dòng tham khảo của bản Stanford (dùng để cảnh báo nếu file tải về bị thiếu/hỏng)
EXPECTED_LINES = {"train": 133317, "dev": 1553, "test": 1268}
SRC, TGT = "en", "vi"


# ----------------------------------------------------------------------------
# 1. Tải dữ liệu
# ----------------------------------------------------------------------------

def _download(url: str, dest: Path, retries: int = 3, timeout: int = 60):
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (envi-nlp-course)"})
            with urllib.request.urlopen(req, timeout=timeout) as r, open(part, "wb") as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
            if part.stat().st_size == 0:
                raise IOError("file rỗng")
            os.replace(part, dest)
            return
        except Exception as e:  # noqa: BLE001
            last_err = e
            print(f"   lần {attempt}/{retries} lỗi khi tải {url}: {e}")
            time.sleep(2 * attempt)
    if part.exists():
        part.unlink()
    raise IOError(f"Không tải được {url}: {last_err}")


def _stanford_complete(d: Path) -> bool:
    for name in STANFORD_SPLITS.values():
        for lang in (SRC, TGT):
            p = d / f"{name}.{lang}"
            if not p.exists() or p.stat().st_size == 0:
                return False
    return True


def fetch_raw(paths: Paths, source: str = "auto") -> dict:
    """Bảo đảm có dữ liệu thô.  Trả về dict mô tả nguồn đã dùng.

    source: "auto" (Stanford rồi dự phòng HF) | "stanford" | "hf"
    """
    sdir = paths.data_raw / "stanford"
    hdir = paths.data_raw / "hf"
    if source in ("auto", "stanford"):
        if _stanford_complete(sdir):
            print("✔ Đã có dữ liệu Stanford trong", sdir)
            return {"source": "stanford", "dir": str(sdir), "dev_from_train": False}
        try:
            print("⬇ Tải IWSLT'15 En-Vi từ Stanford NLP ...")
            for name in STANFORD_SPLITS.values():
                for lang in (SRC, TGT):
                    p = sdir / f"{name}.{lang}"
                    if not p.exists() or p.stat().st_size == 0:
                        _download(STANFORD_BASE + f"{name}.{lang}", p)
            if _stanford_complete(sdir):
                return {"source": "stanford", "dir": str(sdir), "dev_from_train": False}
        except Exception as e:  # noqa: BLE001
            print(f"⚠️  Không tải được từ Stanford ({e}).")
            if source == "stanford":
                raise
    # Dự phòng Hugging Face
    print("⬇ Dùng nguồn dự phòng Hugging Face (parquet) ...")
    import pandas as pd
    from huggingface_hub import hf_hub_download
    hdir.mkdir(parents=True, exist_ok=True)
    for split, fname in HF_FILES.items():
        if (hdir / f"{split}.{SRC}").exists() and (hdir / f"{split}.{TGT}").exists():
            continue
        local = hf_hub_download(HF_REPO, fname, repo_type="dataset", revision=HF_REVISION)
        df = pd.read_parquet(local)
        tr = df["translation"].tolist()
        write_lines(hdir / f"{split}.{SRC}", [t[SRC] for t in tr])
        write_lines(hdir / f"{split}.{TGT}", [t[TGT] for t in tr])
    print("ℹ️  Nguồn HF: tập dev sẽ được tách từ train (vì validation của HF trùng test).")
    return {"source": "hf", "dir": str(hdir), "dev_from_train": True}


def load_raw(info: dict) -> dict:
    d = Path(info["dir"])
    out = {}
    names = STANFORD_SPLITS if info["source"] == "stanford" else {"train": "train", "test": "test"}
    for split, name in names.items():
        src = read_lines(d / f"{name}.{SRC}")
        tgt = read_lines(d / f"{name}.{TGT}")
        # Bỏ các dòng trống ở CUỐI file (thường gặp), rồi mới so số dòng
        while src and tgt and not src[-1].strip() and not tgt[-1].strip():
            src.pop(); tgt.pop()
        if len(src) != len(tgt):
            raise ValueError(f"[{split}] số dòng en ({len(src)}) != vi ({len(tgt)}): file hỏng, hãy xoá {d} và tải lại.")
        exp = EXPECTED_LINES.get(split)
        if exp and info["source"] == "stanford" and abs(len(src) - exp) > 5:
            print(f"⚠️  [{split}] có {len(src)} dòng, khác số tham khảo {exp}. Kiểm tra lại file.")
        out[split] = (src, tgt)
    return out


# ----------------------------------------------------------------------------
# 2. Làm sạch
# ----------------------------------------------------------------------------

_DETOK = {}


def _detok(lang):
    if lang not in _DETOK:
        from sacremoses import MosesDetokenizer
        _DETOK[lang] = MosesDetokenizer(lang=lang)
    return _DETOK[lang]


def clean_line(s: str, lang: str) -> str:
    """Bản Stanford đã được tách từ kiểu Moses và mã hoá HTML (&apos; &quot; ...).

    Ta khôi phục văn bản tự nhiên: giải mã HTML -> ghép lại token (detokenize) -> chuẩn hoá.
    Nhờ vậy, mô hình train từ đầu và mô hình pretrained được đánh giá trên CÙNG một dạng văn bản,
    và sacreBLEU (tự tokenize theo chuẩn 13a) cho điểm so sánh được.
    """
    s = s.strip()
    for _ in range(2):  # một số dòng bị mã hoá 2 lần, ví dụ &amp;apos;
        u = html.unescape(s)
        if u == s:
            break
        s = u
    s = s.replace(" @-@ ", "-").replace(" @,@ ", ",").replace(" @.@ ", ".")
    if s:
        s = _detok(lang).detokenize(s.split())
    return normalize_text(s)


def clean_split(src, tgt, *, is_train: bool, max_words: int = 150, max_ratio: float = 3.5,
                ratio_min_len: int = 4, block_src: set | None = None):
    """Làm sạch một split.  Trả về (src, tgt, báo_cáo_số_câu_bị_loại_theo_lý_do).

    Tập dev/test chỉ loại cặp rỗng (giữ nguyên để so sánh được với các công trình khác).
    Tập train còn lọc thêm: quá dài, tỉ lệ độ dài bất thường, trùng lặp, trùng với dev/test.
    """
    report = Counter()
    out_s, out_t = [], []
    seen = set()
    for s, t in zip(src, tgt):
        s, t = clean_line(s, SRC), clean_line(t, TGT)
        if not s or not t:
            report["rỗng"] += 1
            continue
        if is_train:
            ls, lt = len(s.split()), len(t.split())
            if ls > max_words or lt > max_words:
                report[f"dài > {max_words} từ"] += 1
                continue
            if min(ls, lt) >= ratio_min_len and max(ls, lt) / min(ls, lt) > max_ratio:
                report[f"tỉ lệ độ dài > {max_ratio}"] += 1
                continue
            if (s, t) in seen:
                report["trùng lặp"] += 1
                continue
            if block_src and s in block_src:
                report["trùng câu nguồn dev/test"] += 1
                continue
            seen.add((s, t))
        out_s.append(s)
        out_t.append(t)
    report["giữ lại"] = len(out_s)
    return out_s, out_t, dict(report)


def build_clean_dataset(paths: Paths, raw_info: dict, dev_size_if_carved: int = 1500, seed: int = 42,
                        **filter_kw) -> dict:
    raw = load_raw(raw_info)
    test_s, test_t, rep_test = clean_split(*raw["test"], is_train=False)
    if raw_info["dev_from_train"]:
        dev_s, dev_t, rep_dev = [], [], {}
    else:
        dev_s, dev_t, rep_dev = clean_split(*raw["dev"], is_train=False)
    block = set(test_s) | set(dev_s)
    tr_s, tr_t, rep_train = clean_split(*raw["train"], is_train=True, block_src=block, **filter_kw)

    if raw_info["dev_from_train"]:
        rng = random.Random(seed)
        idx = list(range(len(tr_s)))
        rng.shuffle(idx)
        dev_idx = set(idx[:dev_size_if_carved])
        dev_s = [tr_s[i] for i in sorted(dev_idx)]
        dev_t = [tr_t[i] for i in sorted(dev_idx)]
        dev_src_set = set(dev_s)
        keep = [i for i in range(len(tr_s)) if i not in dev_idx and tr_s[i] not in dev_src_set]
        tr_s = [tr_s[i] for i in keep]
        tr_t = [tr_t[i] for i in keep]
        rep_dev = {"tách từ train": len(dev_s)}

    # Kiểm tra rò rỉ dữ liệu
    check_no_leakage(tr_s, dev_s, test_s, dev_t, test_t)

    for split, (s, t) in {"train": (tr_s, tr_t), "dev": (dev_s, dev_t), "test": (test_s, test_t)}.items():
        write_lines(paths.data_clean / f"{split}.{SRC}", s)
        write_lines(paths.data_clean / f"{split}.{TGT}", t)
    info = {
        "source": raw_info["source"],
        "dev_from_train": raw_info["dev_from_train"],
        "dev_name": "tst2012" if not raw_info["dev_from_train"] else f"{dev_size_if_carved} câu tách từ train",
        "test_name": "tst2013",
        "counts": {"train": len(tr_s), "dev": len(dev_s), "test": len(test_s)},
        "clean_report": {"train": rep_train, "dev": rep_dev, "test": rep_test},
        "filters": {"max_words": filter_kw.get("max_words", 150), "max_ratio": filter_kw.get("max_ratio", 3.5)},
    }
    save_json(info, paths.data_clean / "data_info.json")
    return info


def check_no_leakage(tr_s, dev_s, test_s, dev_t, test_t, max_overlap: float = 0.05):
    test_pairs = set(zip(test_s, test_t))
    dev_pairs = set(zip(dev_s, dev_t))
    if test_pairs and dev_pairs:
        ov = len(test_pairs & dev_pairs) / len(test_pairs)
        if ov > max_overlap:
            raise RuntimeError(f"❌ Dev và test trùng {ov:.0%} — chọn checkpoint trên dev sẽ làm sai lệch kết quả test!")
    tr = set(tr_s)
    n_tr_test = sum(1 for s in test_s if s in tr)
    if n_tr_test:
        raise RuntimeError(f"❌ {n_tr_test} câu nguồn của test còn nằm trong train.")
    print(f"✔ Kiểm tra rò rỉ: dev∩test = {len(test_pairs & dev_pairs)} cặp, train∩test(nguồn) = 0")


def load_clean(paths: Paths, split: str):
    s = read_lines(paths.data_clean / f"{split}.{SRC}")
    t = read_lines(paths.data_clean / f"{split}.{TGT}")
    if len(s) != len(t):
        raise ValueError(f"data_clean/{split}: số dòng en/vi lệch nhau. Chạy lại notebook 01.")
    return s, t


def load_all_clean(paths: Paths, dry_run: bool = False, dry_sizes=(3000, 200, 200)):
    """Đọc train/dev/test đã làm sạch.  DRY_RUN: lấy tập con nhỏ để chạy thử nhanh."""
    if not (paths.data_clean / "data_info.json").exists():
        raise FileNotFoundError("Chưa có dữ liệu sạch. Hãy chạy notebook 01 trước.")
    data = {sp: load_clean(paths, sp) for sp in ("train", "dev", "test")}
    if dry_run:
        for sp, n in zip(("train", "dev", "test"), dry_sizes):
            s, t = data[sp]
            data[sp] = (s[:n], t[:n])
    return data


def length_stats(lines) -> dict:
    lens = np.array([len(x.split()) for x in lines])
    if len(lens) == 0:
        return {}
    return {"n": int(len(lens)), "mean": float(lens.mean()), "median": float(np.median(lens)),
            "p95": float(np.percentile(lens, 95)), "max": int(lens.max()), "min": int(lens.min())}


# ----------------------------------------------------------------------------
# 3. Tokenizer SentencePiece (dùng chung cho 2 mô hình train từ đầu)
# ----------------------------------------------------------------------------

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SEP_PIECE = "<sep>"


def train_spm(paths: Paths, train_src, train_tgt, vocab_size: int = 16000, seed: int = 42,
              max_sentences: int = 1_000_000):
    """Huấn luyện BPE chung (joint) cho tiếng Anh + tiếng Việt.

    - byte_fallback: ký tự lạ được tách thành byte => không bao giờ sinh <unk>.
    - normalization 'identity': văn bản đã chuẩn hoá NFC ở bước làm sạch; không để SentencePiece
      đổi ký tự (NFKC) khiến bản dịch khác tham chiếu.
    - <sep>: token phân cách nguồn/đích cho mô hình GPT (decoder-only).
    """
    import sentencepiece as spm
    paths.spm.mkdir(parents=True, exist_ok=True)
    lines = list(train_src) + list(train_tgt)
    random.Random(seed).shuffle(lines)
    inp = paths.spm / "spm_input.txt"
    write_lines(inp, lines[:max_sentences])
    spm.SentencePieceTrainer.train(
        input=str(inp), model_prefix=str(paths.spm / "spm_envi"), vocab_size=vocab_size,
        model_type="bpe", character_coverage=0.9995, byte_fallback=True,
        pad_id=PAD_ID, unk_id=UNK_ID, bos_id=BOS_ID, eos_id=EOS_ID,
        pad_piece="<pad>", unk_piece="<unk>", bos_piece="<s>", eos_piece="</s>",
        user_defined_symbols=[SEP_PIECE], normalization_rule_name="identity",
        shuffle_input_sentence=True, input_sentence_size=max_sentences, seed_sentencepiece_size=1_000_000,
        num_threads=max(1, os.cpu_count() or 1), hard_vocab_limit=False, minloglevel=2,
    )
    inp.unlink(missing_ok=True)
    return SpmTokenizer(paths.spm_model)


class SpmTokenizer:
    def __init__(self, model_path):
        import sentencepiece as spm
        model_path = Path(model_path)
        if not model_path.exists():
            raise FileNotFoundError(f"Không thấy tokenizer {model_path}. Chạy notebook 01 trước "
                                    "(chú ý DRY_RUN của notebook 01 phải giống notebook hiện tại).")
        self.sp = spm.SentencePieceProcessor(model_file=str(model_path))
        self.pad_id, self.unk_id, self.bos_id, self.eos_id = PAD_ID, UNK_ID, BOS_ID, EOS_ID
        self.sep_id = self.sp.piece_to_id(SEP_PIECE)
        assert self.sp.pad_id() == PAD_ID and self.sp.eos_id() == EOS_ID and self.sp.bos_id() == BOS_ID
        assert self.sep_id != UNK_ID, "Tokenizer thiếu token <sep>"
        self.special = {PAD_ID, BOS_ID, EOS_ID, self.sep_id}
        self.vocab_size = self.sp.get_piece_size()

    def encode(self, texts):
        return self.sp.encode(list(texts), out_type=int)

    def decode(self, ids_list):
        clean = []
        for ids in ids_list:
            if torch.is_tensor(ids):
                ids = ids.tolist()
            out = []
            for i in ids:
                if i == EOS_ID:
                    break
                if i not in self.special:
                    out.append(int(i))
            clean.append(out)
        return [normalize_text(x) for x in self.sp.decode(clean)]


# ----------------------------------------------------------------------------
# 4. Dataset, collate, batching theo số token
# ----------------------------------------------------------------------------

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, src_ids, tgt_ids=None):
        self.src = src_ids
        self.tgt = tgt_ids

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        return i, self.src[i], (self.tgt[i] if self.tgt is not None else None)


def filter_by_token_len(src_ids, tgt_ids, max_src: int, max_tgt: int):
    keep = [i for i, (s, t) in enumerate(zip(src_ids, tgt_ids)) if 0 < len(s) <= max_src and 0 < len(t) <= max_tgt]
    return [src_ids[i] for i in keep], [tgt_ids[i] for i in keep], len(src_ids) - len(keep)


def _pad(seqs, pad, left=False):
    L = max(len(s) for s in seqs)
    out = torch.full((len(seqs), L), pad, dtype=torch.long)
    for i, s in enumerate(seqs):
        if len(s):
            if left:
                out[i, L - len(s):] = torch.as_tensor(s, dtype=torch.long)
            else:
                out[i, :len(s)] = torch.as_tensor(s, dtype=torch.long)
    return out


def seq2seq_collate(batch, pad=PAD_ID, bos=BOS_ID, eos=EOS_ID):
    idx = [b[0] for b in batch]
    src = _pad([list(b[1]) + [eos] for b in batch], pad)
    out = {"idx": torch.tensor(idx), "src": src}
    if batch[0][2] is not None:
        out["tgt_in"] = _pad([[bos] + list(b[2]) for b in batch], pad)
        out["tgt_out"] = _pad([list(b[2]) + [eos] for b in batch], pad)
    return out


IGNORE = -100


def gpt_collate(batch, pad=PAD_ID, bos=BOS_ID, sep=None, eos=EOS_ID):
    """Chuỗi huấn luyện của GPT: <s> nguồn <sep> đích </s>.  Chỉ tính loss trên phần đích."""
    assert sep is not None
    inputs, labels = [], []
    for _, s, t in batch:
        seq = [bos] + list(s) + [sep] + list(t) + [eos]
        inp, lab = seq[:-1], seq[1:]
        lab = [IGNORE] * (len(s) + 1) + lab[len(s) + 1:]
        inputs.append(inp)
        labels.append(lab)
    ids = _pad(inputs, pad)
    lab = _pad(labels, IGNORE)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    for i, x in enumerate(inputs):
        mask[i, :len(x)] = True
    return {"idx": torch.tensor([b[0] for b in batch]), "input_ids": ids, "labels": lab, "attn_mask": mask}


def gpt_prompt_batch(src_list, pad=PAD_ID, bos=BOS_ID, sep=None):
    """Prompt sinh bản dịch: <s> nguồn <sep>, đệm TRÁI để mọi câu kết thúc cùng vị trí."""
    seqs = [[bos] + list(s) + [sep] for s in src_list]
    ids = _pad(seqs, pad, left=True)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    L = ids.size(1)
    for i, s in enumerate(seqs):
        mask[i, L - len(s):] = True
    return ids, mask


class TokenBatchSampler:
    """Gom câu có độ dài gần nhau vào cùng batch, giới hạn tổng số token (max_tokens).

    Tiết kiệm bộ nhớ và thời gian so với batch cố định số câu.  Thứ tự batch được xáo trộn
    theo seed + epoch => khi resume từ checkpoint, ta tái tạo ĐÚNG danh sách batch của epoch đó.
    """

    def __init__(self, lengths, max_tokens: int, shuffle: bool = True, seed: int = 0,
                 max_sentences: int | None = None):
        self.lengths = np.asarray(lengths)
        self.max_tokens = max_tokens
        self.shuffle = shuffle
        self.seed = seed
        self.max_sentences = max_sentences

    def batches(self, epoch: int = 0):
        rng = np.random.RandomState(self.seed + epoch)
        noise = rng.uniform(0, 1, len(self.lengths)) if self.shuffle else np.zeros(len(self.lengths))
        order = np.lexsort((noise, self.lengths))
        batches, cur, cur_max = [], [], 0
        for i in order:
            L = int(self.lengths[i])
            new_max = max(cur_max, L)
            too_many = self.max_sentences is not None and len(cur) + 1 > self.max_sentences
            if cur and (new_max * (len(cur) + 1) > self.max_tokens or too_many):
                batches.append(cur)
                cur, new_max = [], L
            cur.append(int(i))
            cur_max = new_max
        if cur:
            batches.append(cur)
        if self.shuffle:
            rng.shuffle(batches)
        return batches


def sorted_eval_batches(lengths, batch_size: int):
    """Batch cho suy luận: sắp theo độ dài (nhanh hơn), trả về danh sách chỉ số gốc."""
    order = np.argsort(np.asarray(lengths), kind="stable")[::-1]
    return [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]

Overwriting envi_lib/data.py


In [6]:
%%writefile envi_lib/models.py
"""Cài đặt Transformer từ đầu (PyTorch thuần) cho 2 kiến trúc:

  - TransformerSeq2Seq : Encoder-Decoder (Vaswani et al., 2017)
  - GPTDecoder         : Decoder-only (kiểu GPT), dịch máy dưới dạng mô hình ngôn ngữ có điều kiện

Hai mô hình dùng CHUNG các khối (MultiHeadAttention, FeedForward, PositionalEncoding) để phép
so sánh kiến trúc là công bằng.

Quy ước mặt nạ (mask): tensor bool, True = ĐƯỢC PHÉP chú ý.
Luôn cho mỗi vị trí chú ý tới chính nó (đường chéo) để không có hàng nào bị che toàn bộ
-> tránh NaN khi câu bị đệm trái (left padding) lúc sinh văn bản với GPT.
"""
from __future__ import annotations

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Các khối dùng chung
# ----------------------------------------------------------------------------

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention:  Attention(Q,K,V) = softmax(QK^T / sqrt(d_k) + mask) V"""

    def __init__(self, d_model: int, n_heads: int, attn_dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model phải chia hết cho n_heads"
        self.h, self.dk = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = attn_dropout
        self.last_attn = None  # lưu trọng số attention khi cần trực quan hoá

    def forward(self, x_q, x_kv, mask=None, need_weights: bool = False):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, Lq, self.h, self.dk).transpose(1, 2)   # (B,h,Lq,dk)
        k = self.k_proj(x_kv).view(B, Lk, self.h, self.dk).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Lk, self.h, self.dk).transpose(1, 2)
        p = self.attn_dropout if self.training else 0.0
        if need_weights:
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
            if mask is not None:
                scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
            w = torch.softmax(scores.float(), dim=-1).to(q.dtype)
            self.last_attn = w.detach()
            out = F.dropout(w, p, self.training) @ v
        else:
            # Cài đặt tối ưu (FlashAttention / memory-efficient) của PyTorch — kết quả tương đương
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p)
        out = out.transpose(1, 2).reshape(B, Lq, D)
        return self.o_proj(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout, activation="relu"):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.act = nn.ReLU() if activation == "relu" else nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.drop(self.act(self.fc1(x))))


class PositionalEncoding(nn.Module):
    """Mã hoá vị trí: 'sinusoidal' (cố định, như bài báo gốc) hoặc 'learned' (học được, như GPT)."""

    def __init__(self, d_model, max_len=1024, kind="sinusoidal"):
        super().__init__()
        self.kind = kind
        self.max_len = max_len
        if kind == "sinusoidal":
            pe = torch.zeros(max_len, d_model)
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(pos * div)
            pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe, persistent=False)
        elif kind == "learned":
            self.emb = nn.Embedding(max_len, d_model)
            nn.init.normal_(self.emb.weight, std=0.02)
        else:
            raise ValueError(kind)

    def forward(self, position_ids):
        if int(position_ids.max()) >= self.max_len:
            raise ValueError(f"Chuỗi dài {int(position_ids.max()) + 1} vượt max_len={self.max_len}")
        return self.pe[position_ids] if self.kind == "sinusoidal" else self.emb(position_ids)


class Block(nn.Module):
    """Một lớp Transformer: self-attention [+ cross-attention] + FFN, có residual và LayerNorm.

    pre_norm=True  : x = x + Sublayer(LN(x))  (ổn định hơn, ít phụ thuộc warmup)
    pre_norm=False : x = LN(x + Sublayer(x))  (Post-LN, đúng bài báo gốc)
    """

    def __init__(self, d_model, n_heads, d_ff, dropout, attn_dropout, cross: bool, pre_norm: bool,
                 activation="relu"):
        super().__init__()
        self.pre_norm = pre_norm
        self.self_attn = MultiHeadAttention(d_model, n_heads, attn_dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, attn_dropout) if cross else None
        self.ln2 = nn.LayerNorm(d_model) if cross else None
        self.ffn = FeedForward(d_model, d_ff, dropout, activation)
        self.ln3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def _sub(self, x, ln, fn):
        if self.pre_norm:
            return x + self.drop(fn(ln(x)))
        return ln(x + self.drop(fn(x)))

    def forward(self, x, self_mask, memory=None, cross_mask=None, need_weights=False):
        x = self._sub(x, self.ln1, lambda h: self.self_attn(h, h, self_mask, need_weights))
        if self.cross_attn is not None:
            x = self._sub(x, self.ln2, lambda h: self.cross_attn(h, memory, cross_mask, need_weights))
        x = self._sub(x, self.ln3, self.ffn)
        return x


def _diag(L, device):
    return torch.eye(L, dtype=torch.bool, device=device)


def causal_mask(L, device):
    return torch.ones(L, L, dtype=torch.bool, device=device).tril()


def _init_weights(module):
    for m in module.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


# ----------------------------------------------------------------------------
# 1. Encoder-Decoder
# ----------------------------------------------------------------------------

@dataclass
class Seq2SeqConfig:
    vocab_size: int = 16000
    d_model: int = 512
    n_heads: int = 4
    d_ff: int = 1024
    enc_layers: int = 6
    dec_layers: int = 6
    dropout: float = 0.3
    attn_dropout: float = 0.1
    activation: str = "relu"
    pre_norm: bool = True
    pos_encoding: str = "sinusoidal"
    max_len: int = 512
    tie_embeddings: bool = True
    pad_id: int = 0

    def to_dict(self):
        return asdict(self)


class TransformerSeq2Seq(nn.Module):
    def __init__(self, cfg: Seq2SeqConfig):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        # Từ vựng chung Anh-Việt => dùng chung 1 bảng embedding cho encoder, decoder và lớp đầu ra
        self.embed = nn.Embedding(cfg.vocab_size, d, padding_idx=cfg.pad_id)
        nn.init.normal_(self.embed.weight, mean=0, std=d ** -0.5)
        with torch.no_grad():
            self.embed.weight[cfg.pad_id].zero_()
        self.pos = PositionalEncoding(d, cfg.max_len, cfg.pos_encoding)
        self.drop = nn.Dropout(cfg.dropout)
        blk = lambda cross: Block(d, cfg.n_heads, cfg.d_ff, cfg.dropout, cfg.attn_dropout, cross,
                                  cfg.pre_norm, cfg.activation)
        self.encoder = nn.ModuleList([blk(False) for _ in range(cfg.enc_layers)])
        self.decoder = nn.ModuleList([blk(True) for _ in range(cfg.dec_layers)])
        self.enc_ln = nn.LayerNorm(d) if cfg.pre_norm else nn.Identity()
        self.dec_ln = nn.LayerNorm(d) if cfg.pre_norm else nn.Identity()
        if not cfg.tie_embeddings:
            self.out_proj = nn.Linear(d, cfg.vocab_size, bias=False)
        _init_weights(self.encoder)
        _init_weights(self.decoder)

    def _embed(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device).unsqueeze(0).expand_as(ids)
        return self.drop(self.embed(ids) * math.sqrt(self.cfg.d_model) + self.pos(pos))

    def encode(self, src, need_weights=False):
        key_ok = src != self.cfg.pad_id                                   # (B,Ls)
        self_mask = key_ok[:, None, None, :] | _diag(src.size(1), src.device)
        x = self._embed(src)
        for layer in self.encoder:
            x = layer(x, self_mask, need_weights=need_weights)
        return self.enc_ln(x), key_ok

    def decode(self, tgt_in, memory, src_key_ok, need_weights=False):
        Lt = tgt_in.size(1)
        tgt_ok = tgt_in != self.cfg.pad_id
        self_mask = (causal_mask(Lt, tgt_in.device)[None, None] & tgt_ok[:, None, None, :]) | _diag(Lt, tgt_in.device)
        cross_mask = src_key_ok[:, None, None, :]
        x = self._embed(tgt_in)
        for layer in self.decoder:
            x = layer(x, self_mask, memory, cross_mask, need_weights=need_weights)
        x = self.dec_ln(x)
        if self.cfg.tie_embeddings:
            return F.linear(x, self.embed.weight)
        return self.out_proj(x)

    def forward(self, src, tgt_in):
        memory, key_ok = self.encode(src)
        return self.decode(tgt_in, memory, key_ok)


# ----------------------------------------------------------------------------
# 2. Decoder-only (GPT)
# ----------------------------------------------------------------------------

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    d_model: int = 512
    n_heads: int = 4
    d_ff: int = 1024
    n_layers: int = 14
    dropout: float = 0.3
    attn_dropout: float = 0.1
    activation: str = "relu"
    pre_norm: bool = True
    pos_encoding: str = "learned"
    max_len: int = 512
    tie_embeddings: bool = True
    pad_id: int = 0

    def to_dict(self):
        return asdict(self)


class GPTDecoder(nn.Module):
    """Chuỗi đầu vào: <s> câu_nguồn <sep> câu_đích </s>.  Che tương lai (causal mask)."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.embed = nn.Embedding(cfg.vocab_size, d, padding_idx=cfg.pad_id)
        nn.init.normal_(self.embed.weight, mean=0, std=d ** -0.5)
        with torch.no_grad():
            self.embed.weight[cfg.pad_id].zero_()
        self.pos = PositionalEncoding(d, cfg.max_len, cfg.pos_encoding)
        self.drop = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([
            Block(d, cfg.n_heads, cfg.d_ff, cfg.dropout, cfg.attn_dropout, False, cfg.pre_norm, cfg.activation)
            for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(d) if cfg.pre_norm else nn.Identity()
        if not cfg.tie_embeddings:
            self.out_proj = nn.Linear(d, cfg.vocab_size, bias=False)
        _init_weights(self.layers)

    @staticmethod
    def position_ids_from_mask(attn_mask):
        """Vị trí tính từ token thật đầu tiên => đệm trái hay phải đều cho cùng vị trí."""
        return (attn_mask.long().cumsum(-1) - 1).clamp(min=0)

    def forward(self, input_ids, attn_mask=None, need_weights=False):
        B, L = input_ids.shape
        if attn_mask is None:
            attn_mask = input_ids != self.cfg.pad_id
        pos = self.position_ids_from_mask(attn_mask)
        x = self.drop(self.embed(input_ids) * math.sqrt(self.cfg.d_model) + self.pos(pos))
        mask = (causal_mask(L, input_ids.device)[None, None] & attn_mask[:, None, None, :]) | _diag(L, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask, need_weights=need_weights)
        x = self.ln_f(x)
        if self.cfg.tie_embeddings:
            return F.linear(x, self.embed.weight)
        return self.out_proj(x)


# ----------------------------------------------------------------------------
# Tiện ích
# ----------------------------------------------------------------------------

def gpt_layers_to_match(target_params: int, base: GPTConfig, max_layers: int = 40) -> int:
    """Chọn số lớp GPT để tổng tham số gần nhất với mô hình Encoder-Decoder (so sánh công bằng)."""
    best, best_diff = 1, None
    for n in range(1, max_layers + 1):
        cfg = GPTConfig(**{**base.to_dict(), "n_layers": n})
        p = sum(x.numel() for x in GPTDecoder(cfg).parameters())
        diff = abs(p - target_params)
        if best_diff is None or diff < best_diff:
            best, best_diff = n, diff
    return best


def label_smoothed_ce(logits, target, eps: float = 0.1, ignore_index: int = -100):
    """Cross-entropy có label smoothing (Szegedy et al., 2016), trung bình trên các token hợp lệ.

    Trả về (loss_trung_bình, nll_trung_bình, số_token).  nll dùng để tính perplexity.
    """
    V = logits.size(-1)
    logits = logits.reshape(-1, V).float()
    target = target.reshape(-1)
    valid = target != ignore_index
    n = int(valid.sum())
    if n == 0:
        z = logits.sum() * 0.0
        return z, z.detach(), 0
    lp = torch.log_softmax(logits[valid], dim=-1)
    t = target[valid]
    nll = -lp.gather(1, t.unsqueeze(1)).squeeze(1)
    smooth = -lp.mean(dim=-1)
    loss = ((1 - eps) * nll + eps * smooth).mean()
    return loss, nll.mean().detach(), n

Overwriting envi_lib/models.py


In [7]:
%%writefile envi_lib/decoding.py
"""Giải mã (sinh bản dịch) cho 2 mô hình tự cài: greedy và beam search.

beam_size = 1 trong beam search phải cho kết quả GIỐNG HỆT greedy — đây là một bài kiểm thử
tự động trong selftest.py.
"""
from __future__ import annotations

import torch

from .common import autocast_ctx
from .data import BOS_ID, EOS_ID, PAD_ID, gpt_prompt_batch, seq2seq_collate, sorted_eval_batches


# ----------------------------------------------------------------------------
# Lõi beam search dùng chung
# ----------------------------------------------------------------------------

@torch.no_grad()
def _beam_core(next_logprobs, batch_size, beam_size, max_new_tokens, device,
               eos_id=EOS_ID, pad_id=PAD_ID, length_penalty=1.0, min_new_tokens=1):
    """next_logprobs(gen: LongTensor (B*K, t)) -> FloatTensor (B*K, V) log-xác suất token kế tiếp.

    gen là phần ĐÃ SINH (không gồm prompt); hàm next_logprobs tự ghép với prompt/memory.
    Điểm chuẩn hoá theo độ dài: score / len^length_penalty (như fairseq/HF).
    """
    B, K = batch_size, beam_size
    beam_scores = torch.zeros(B, K, device=device)
    beam_scores[:, 1:] = float("-inf")          # bước đầu: chỉ mở rộng từ beam 0
    gen = torch.empty(B * K, 0, dtype=torch.long, device=device)
    finished = [[] for _ in range(B)]           # (điểm_chuẩn_hoá, list token)
    done = [False] * B

    for step in range(max_new_tokens):
        lp = next_logprobs(gen).float()          # (B*K, V)
        V = lp.size(-1)
        if step < min_new_tokens:
            lp[:, eos_id] = float("-inf")
        lp[:, pad_id] = float("-inf")
        cand = (beam_scores.view(-1, 1) + lp).view(B, K * V)
        top_s, top_i = cand.topk(2 * K, dim=-1)  # lấy 2K để còn đủ K beam sau khi tách beam kết thúc
        top_beam, top_tok = top_i // V, top_i % V

        new_scores = torch.full((B, K), float("-inf"), device=device)
        new_src = torch.zeros(B, K, dtype=torch.long, device=device)
        new_tok = torch.full((B, K), pad_id, dtype=torch.long, device=device)
        top_s_l, top_beam_l, top_tok_l = top_s.tolist(), top_beam.tolist(), top_tok.tolist()
        for b in range(B):
            if done[b]:
                new_src[b] = torch.arange(K, device=device)   # giữ nguyên, điền pad
                continue
            j = 0
            for s, kb, tok in zip(top_s_l[b], top_beam_l[b], top_tok_l[b]):
                if s == float("-inf"):
                    break
                if tok == eos_id:
                    seq = gen[b * K + kb].tolist()
                    finished[b].append((s / ((step + 1) ** length_penalty), seq))
                else:
                    new_scores[b, j] = s
                    new_src[b, j] = kb
                    new_tok[b, j] = tok
                    j += 1
                if j == K:
                    break
            # Điều kiện dừng (heuristic giống HF early_stopping=False)
            if len(finished[b]) >= K:
                finished[b].sort(key=lambda x: -x[0])
                finished[b] = finished[b][:K]
                best_active = new_scores[b, 0].item() / ((step + 1) ** length_penalty)
                if best_active <= finished[b][-1][0]:
                    done[b] = True
        if all(done):
            break
        flat_src = (torch.arange(B, device=device).unsqueeze(1) * K + new_src).view(-1)
        gen = torch.cat([gen[flat_src], new_tok.view(-1, 1)], dim=1)
        beam_scores = new_scores

    results = []
    for b in range(B):
        if not done[b]:
            for k in range(K):
                s = beam_scores[b, k].item()
                if s != float("-inf"):
                    finished[b].append((s / (gen.size(1) ** length_penalty), gen[b * K + k].tolist()))
        best = max(finished[b], key=lambda x: x[0])[1] if finished[b] else []
        results.append([t for t in best if t != pad_id])
    return results


@torch.no_grad()
def _greedy_core(next_logprobs, batch_size, max_new_tokens, device, eos_id=EOS_ID, pad_id=PAD_ID,
                 min_new_tokens=1):
    gen = torch.empty(batch_size, 0, dtype=torch.long, device=device)
    alive = torch.ones(batch_size, dtype=torch.bool, device=device)
    for step in range(max_new_tokens):
        lp = next_logprobs(gen).float()
        if step < min_new_tokens:
            lp[:, eos_id] = float("-inf")
        lp[:, pad_id] = float("-inf")
        tok = lp.argmax(-1)
        tok = torch.where(alive, tok, torch.full_like(tok, pad_id))
        gen = torch.cat([gen, tok.unsqueeze(1)], dim=1)
        alive &= tok != eos_id
        if not alive.any():
            break
    out = []
    for row in gen.tolist():
        seq = []
        for t in row:
            if t == eos_id or t == pad_id:
                break
            seq.append(t)
        out.append(seq)
    return out


def _force_eos_after_limit(lp, gen_len, limits, eos_id=EOS_ID):
    """Mỗi câu có giới hạn độ dài riêng (theo độ dài câu nguồn): quá giới hạn thì buộc sinh </s>.

    Nhờ vậy kết quả dịch một câu KHÔNG phụ thuộc việc nó nằm chung batch với câu dài hay ngắn.
    """
    over = limits[: lp.size(0)] <= gen_len
    if over.any():
        lp[over] = float("-inf")
        lp[over, eos_id] = 0.0
    return lp


# ----------------------------------------------------------------------------
# Encoder-Decoder
# ----------------------------------------------------------------------------

@torch.no_grad()
def seq2seq_generate_batch(model, src, beam_size=1, max_len_a=1.5, max_len_b=10, length_penalty=1.0,
                           amp_dtype=None):
    """src: (B, Ls) đã có </s>, đệm phải.  Trả về list token id (không gồm </s>)."""
    model.eval()
    device = src.device
    B = src.size(0)
    src_lens = (src != PAD_ID).sum(1)
    limits = (src_lens.float() * max_len_a + max_len_b).long().clamp(max=model.cfg.max_len - 2)
    max_new = int(limits.max()) + 1
    with autocast_ctx(amp_dtype):
        memory, key_ok = model.encode(src)
    K = beam_size
    if K > 1:
        memory = memory.repeat_interleave(K, dim=0)
        key_ok = key_ok.repeat_interleave(K, dim=0)
        limits = limits.repeat_interleave(K, dim=0)
    bos = torch.full((B * K, 1), BOS_ID, dtype=torch.long, device=device)

    def next_lp(gen):
        tgt = torch.cat([bos[: gen.size(0)], gen], dim=1)
        with autocast_ctx(amp_dtype):
            logits = model.decode(tgt, memory[: gen.size(0)], key_ok[: gen.size(0)])
        lp = torch.log_softmax(logits[:, -1].float(), dim=-1)
        return _force_eos_after_limit(lp, gen.size(1), limits)

    if K == 1:
        return _greedy_core(next_lp, B, max_new, device)
    return _beam_core(next_lp, B, K, max_new, device, length_penalty=length_penalty)


# ----------------------------------------------------------------------------
# Decoder-only (GPT)
# ----------------------------------------------------------------------------

@torch.no_grad()
def gpt_generate_batch(model, src_ids_list, sep_id, beam_size=1, max_len_a=1.5, max_len_b=10,
                       length_penalty=1.0, amp_dtype=None, device=None):
    model.eval()
    device = device or next(model.parameters()).device
    prompt, pmask = gpt_prompt_batch(src_ids_list, sep=sep_id)
    prompt, pmask = prompt.to(device), pmask.to(device)
    B = prompt.size(0)
    room = model.cfg.max_len - prompt.size(1) - 1
    if room <= 1:
        raise ValueError("Câu nguồn quá dài so với max_len của GPT.")
    limits = torch.tensor([min(int(len(s) * max_len_a + max_len_b), room) for s in src_ids_list], device=device)
    max_new = int(limits.max()) + 1
    K = beam_size
    if K > 1:
        prompt = prompt.repeat_interleave(K, dim=0)
        pmask = pmask.repeat_interleave(K, dim=0)
        limits = limits.repeat_interleave(K, dim=0)

    def next_lp(gen):
        n = gen.size(0)
        ids = torch.cat([prompt[:n], gen], dim=1)
        mask = torch.cat([pmask[:n], torch.ones_like(gen, dtype=torch.bool)], dim=1)
        with autocast_ctx(amp_dtype):
            logits = model(ids, mask)
        lp = torch.log_softmax(logits[:, -1].float(), dim=-1)
        return _force_eos_after_limit(lp, gen.size(1), limits)

    if K == 1:
        return _greedy_core(next_lp, B, max_new, device)
    return _beam_core(next_lp, B, K, max_new, device, length_penalty=length_penalty)


# ----------------------------------------------------------------------------
# Dịch cả một tập câu (có sắp xếp theo độ dài để nhanh, rồi trả về đúng thứ tự gốc)
# ----------------------------------------------------------------------------

def translate_corpus(model, tokenizer, src_texts, kind: str, beam_size=1, batch_size=64,
                     amp_dtype=None, length_penalty=1.0, max_src_tokens=None, progress=True):
    """kind: 'seq2seq' hoặc 'gpt'.  Trả về list câu tiếng Việt cùng thứ tự với src_texts."""
    from tqdm.auto import tqdm
    device = next(model.parameters()).device
    src_ids = tokenizer.encode(src_texts)
    if max_src_tokens:
        n_trunc = sum(1 for s in src_ids if len(s) > max_src_tokens)
        if n_trunc:
            print(f"⚠️  {n_trunc} câu nguồn dài hơn {max_src_tokens} token bị cắt bớt khi dịch.")
        src_ids = [s[:max_src_tokens] for s in src_ids]
    bs = max(1, batch_size // beam_size) if beam_size > 1 else batch_size
    batches = sorted_eval_batches([len(s) for s in src_ids], bs)
    out = [None] * len(src_ids)
    it = tqdm(batches, desc=f"Dịch ({kind}, beam={beam_size})", leave=False) if progress else batches
    for idx in it:
        if kind == "seq2seq":
            batch = seq2seq_collate([(i, src_ids[i], None) for i in idx])
            ids = seq2seq_generate_batch(model, batch["src"].to(device), beam_size=beam_size,
                                         length_penalty=length_penalty, amp_dtype=amp_dtype)
        elif kind == "gpt":
            ids = gpt_generate_batch(model, [src_ids[i] for i in idx], tokenizer.sep_id, beam_size=beam_size,
                                     length_penalty=length_penalty, amp_dtype=amp_dtype, device=device)
        else:
            raise ValueError(kind)
        for i, h in zip(idx, tokenizer.decode(ids)):
            out[i] = h
    assert all(o is not None for o in out)
    return out

Overwriting envi_lib/decoding.py


In [8]:
%%writefile envi_lib/train.py
"""Vòng lặp huấn luyện dùng chung cho MỌI mô hình (tự cài, HF seq2seq, LLM + LoRA).

Các cơ chế phòng rủi ro:
  * Checkpoint 'last.pt' lưu định kỳ (model + optimizer + scheduler + scaler + RNG + vị trí batch).
    Colab mất kết nối => chạy lại ô huấn luyện là tự động tiếp tục ĐÚNG chỗ cũ.
  * Ghi file tạm rồi đổi tên => không bao giờ có checkpoint hỏng nửa chừng.
  * Lưu 'best' theo điểm dev (BLEU) + early stopping.
  * Phát hiện loss NaN/inf: bỏ qua bước đó; quá nhiều lần liên tiếp => dừng và báo nguyên nhân.
  * Bắt lỗi hết bộ nhớ GPU (OOM): bỏ qua batch; lặp lại nhiều lần => dừng và gợi ý giảm max_tokens.
  * Giới hạn thời gian (time_budget_hours): dừng an toàn trước khi hết quota Colab.
  * Ngắt tay (Interrupt): vẫn lưu checkpoint trước khi dừng.
"""
from __future__ import annotations

import csv
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Callable

import numpy as np
import torch

from .common import LIB_VERSION, autocast_ctx, fmt_seconds, now_str, save_json


@dataclass
class TrainConfig:
    epochs: int = 50
    max_tokens: int = 8192            # số token tối đa / batch (trước tích luỹ gradient)
    max_sentences: int | None = None  # giới hạn số câu / batch (tuỳ chọn)
    grad_accum: int = 1
    lr: float = 7e-4
    warmup_steps: int = 4000
    schedule: str = "inverse_sqrt"    # inverse_sqrt | linear | cosine | constant
    min_lr_ratio: float = 0.0         # cho linear/cosine
    weight_decay: float = 0.0001
    betas: tuple = (0.9, 0.98)
    adam_eps: float = 1e-8
    clip_norm: float = 1.0
    label_smoothing: float = 0.1
    patience: int = 6                 # số lần đánh giá liên tiếp không cải thiện thì dừng
    eval_every_steps: int | None = None  # None = đánh giá cuối mỗi epoch
    save_every_steps: int = 1000      # lưu last.pt định kỳ giữa epoch
    log_every_steps: int = 200
    max_steps: int | None = None
    time_budget_hours: float | None = None
    seed: int = 42
    amp: str = "auto"                 # auto | bf16 | fp16 | fp32
    save_optimizer: bool = True

    def to_dict(self):
        d = asdict(self)
        d["betas"] = list(self.betas)
        return d


def make_scheduler(optimizer, cfg: TrainConfig, total_steps: int):
    w = max(1, cfg.warmup_steps)

    def f(step):
        step = max(1, step)
        if cfg.schedule == "inverse_sqrt":
            return min(step / w, math.sqrt(w / step))
        if step < w:
            return step / w
        if cfg.schedule == "constant":
            return 1.0
        prog = min(1.0, (step - w) / max(1, total_steps - w))
        if cfg.schedule == "linear":
            return max(cfg.min_lr_ratio, 1 - prog * (1 - cfg.min_lr_ratio))
        if cfg.schedule == "cosine":
            return cfg.min_lr_ratio + (1 - cfg.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * prog))
        raise ValueError(cfg.schedule)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


def _atomic_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _rng_state():
    st = {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st


def _set_rng_state(st):
    try:
        random.setstate(st["py"])
        np.random.set_state(st["np"])
        torch.set_rng_state(st["torch"])
        if "cuda" in st and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(st["cuda"])
    except Exception as e:  # noqa: BLE001
        print(f"⚠️  Không khôi phục được trạng thái ngẫu nhiên ({e}) — vẫn tiếp tục.")


class TooManyFailures(RuntimeError):
    pass


class Trainer:
    """
    loss_fn(model, batch, cfg) -> (loss, nll_mean, n_tokens)   [batch đã ở trên device]
    eval_fn(model) -> dict, bắt buộc có khoá 'score' (càng cao càng tốt, ví dụ BLEU dev)
    save_best_fn(model, dir) -> None : tuỳ chọn, cách lưu mô hình tốt nhất (mặc định state_dict)
    """

    def __init__(self, model, dataset, collate_fn, lengths, cfg: TrainConfig, ckpt_dir,
                 loss_fn: Callable, eval_fn: Callable | None = None, device=None,
                 save_best_fn: Callable | None = None, run_name: str = "run",
                 extra_meta: dict | None = None, params=None, last_dir=None):
        self.model = model
        self.dataset = dataset
        self.collate_fn = collate_fn
        self.lengths = lengths
        self.cfg = cfg
        self.ckpt_dir = Path(ckpt_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        # last.pt (kèm trạng thái optimizer, có thể nặng vài GB) có thể đặt ở ổ máy ảo thay vì Drive
        self.last_dir = Path(last_dir) if last_dir else self.ckpt_dir
        self.last_dir.mkdir(parents=True, exist_ok=True)
        self.loss_fn = loss_fn
        self.eval_fn = eval_fn
        self.device = device or next(model.parameters()).device
        self.save_best_fn = save_best_fn
        self.run_name = run_name
        self.extra_meta = extra_meta or {}
        from .common import pick_amp_dtype
        self.amp_dtype = pick_amp_dtype(cfg.amp) if self.device.type == "cuda" else None
        self.use_scaler = self.amp_dtype == torch.float16
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_scaler)

        from .data import TokenBatchSampler
        self.sampler = TokenBatchSampler(lengths, cfg.max_tokens, shuffle=True, seed=cfg.seed,
                                         max_sentences=cfg.max_sentences)
        self.steps_per_epoch = math.ceil(len(self.sampler.batches(0)) / cfg.grad_accum)
        self.total_steps = cfg.max_steps or self.steps_per_epoch * cfg.epochs

        params = [p for p in (params if params is not None else model.parameters()) if p.requires_grad]
        decay = [p for p in params if p.ndim >= 2]
        no_decay = [p for p in params if p.ndim < 2]
        self.optimizer = torch.optim.AdamW(
            [{"params": decay, "weight_decay": cfg.weight_decay}, {"params": no_decay, "weight_decay": 0.0}],
            lr=cfg.lr, betas=tuple(cfg.betas), eps=cfg.adam_eps)
        self.scheduler = make_scheduler(self.optimizer, cfg, self.total_steps)

        self.state = {"epoch": 0, "batch_in_epoch": 0, "global_step": 0, "best_score": -float("inf"),
                      "best_step": 0, "bad_evals": 0, "train_seconds": 0.0, "finished": False}
        self.history = []

    # ---------------------------------------------------------------- checkpoint
    @property
    def last_path(self):
        return self.last_dir / "last.pt"

    def _trainable_state(self):
        # Với LoRA chỉ lưu tham số được huấn luyện (vài chục MB thay vì vài GB)
        return {k: v.detach().cpu() for k, v in self.model.state_dict().items()
                if k in self._trainable_names}

    @property
    def _trainable_names(self):
        if not hasattr(self, "_tn"):
            names = {n for n, p in self.model.named_parameters() if p.requires_grad}
            all_params = {n for n, _ in self.model.named_parameters()}
            # Nếu train toàn bộ thì lưu cả state_dict (gồm buffer)
            self._tn = None if names == all_params else names
        return self._tn

    def save_last(self):
        sd = self.model.state_dict() if self._trainable_names is None else self._trainable_state()
        ck = {"model": sd, "state": self.state, "history": self.history, "rng": _rng_state(),
              "cfg": self.cfg.to_dict(), "lib_version": LIB_VERSION, "saved_at": now_str(),
              "scheduler": self.scheduler.state_dict(), "scaler": self.scaler.state_dict()}
        if self.cfg.save_optimizer:
            ck["optimizer"] = self.optimizer.state_dict()
        _atomic_save(ck, self.last_path)

    def try_resume(self) -> bool:
        if not self.last_path.exists():
            return False
        ck = torch.load(self.last_path, map_location="cpu", weights_only=False)
        old = ck.get("cfg", {})
        important = ["lr", "max_tokens", "grad_accum", "schedule", "warmup_steps", "seed", "epochs"]
        diffs = {k: (old.get(k), self.cfg.to_dict().get(k)) for k in important if old.get(k) != self.cfg.to_dict().get(k)}
        if diffs:
            print(f"⚠️  Cấu hình khác với checkpoint cũ: {diffs}. Tiếp tục với cấu hình MỚI "
                  "(nếu muốn huấn luyện lại từ đầu: xoá thư mục checkpoint của run này).")
        try:
            missing, unexpected = self.model.load_state_dict(ck["model"], strict=self._trainable_names is None)
        except RuntimeError as e:
            raise RuntimeError(
                f"Checkpoint {self.last_path} không khớp với mô hình hiện tại (thường do đã đổi mô hình/cấu hình "
                f"nhưng còn checkpoint cũ). Xoá file đó (và thư mục {self.ckpt_dir}) rồi chạy lại.\n{str(e)[:300]}") from None
        if self._trainable_names is not None and unexpected:
            raise RuntimeError(f"Checkpoint có tham số lạ: {unexpected[:5]}")
        if "optimizer" in ck:
            self.optimizer.load_state_dict(ck["optimizer"])
        else:
            print("ℹ️  Checkpoint không có trạng thái optimizer — khởi tạo lại optimizer.")
        self.scheduler.load_state_dict(ck["scheduler"])
        if self.use_scaler and ck.get("scaler"):
            self.scaler.load_state_dict(ck["scaler"])
        self.state.update(ck["state"])
        self.history = ck.get("history", [])
        _set_rng_state(ck.get("rng", {}))
        print(f"🔁 Tiếp tục từ checkpoint ({ck.get('saved_at')}): epoch {self.state['epoch'] + 1}, "
              f"batch {self.state['batch_in_epoch']}, step {self.state['global_step']}, "
              f"best dev = {self.state['best_score']:.2f}")
        return True

    def _save_best(self):
        if self.save_best_fn is not None:
            self.save_best_fn(self.model, self.ckpt_dir / "best")
        else:
            _atomic_save({"model": self.model.state_dict(), "step": self.state["global_step"],
                          "score": self.state["best_score"], "meta": self.extra_meta}, self.ckpt_dir / "best.pt")

    def _log_history(self, row):
        self.history.append(row)
        path = self.ckpt_dir / "history.csv"
        keys = sorted({k for r in self.history for k in r})
        tmp = path.with_suffix(".csv.tmp")
        with open(tmp, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(self.history)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- evaluate
    def evaluate(self, tag=""):
        row = {"step": self.state["global_step"], "epoch": self.state["epoch"],
               "lr": self.optimizer.param_groups[0]["lr"], "time": now_str()}
        self.state["last_eval_step"] = self.state["global_step"]
        if self.eval_fn is not None:
            self.model.eval()
            res = self.eval_fn(self.model)
            self.model.train()
            row.update(res)
            score = res["score"]
            improved = score > self.state["best_score"] + 1e-6
            if improved:
                self.state["best_score"], self.state["best_step"], self.state["bad_evals"] = score, self.state["global_step"], 0
                self._save_best()
            else:
                self.state["bad_evals"] += 1
            row["best"] = improved
            extra = " ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in res.items() if k != "score")
            if improved:
                note = "⭐ tốt nhất mới"
            else:
                note = "(không cải thiện %d/%d)" % (self.state["bad_evals"], self.cfg.patience)
            print(f"   📊 {tag} step {self.state['global_step']}: dev score={score:.2f} {extra} {note}")
        self._log_history(row)

    # ---------------------------------------------------------------- train
    def train(self):
        from tqdm.auto import tqdm
        cfg = self.cfg
        self.try_resume()
        if self.state["finished"]:
            reason = self.state.get("stop_reason")
            if reason == "đủ số epoch" and self.state["epoch"] < cfg.epochs:
                print(f"➕ Tăng số epoch lên {cfg.epochs}: huấn luyện tiếp.")
                self.state["finished"] = False
            elif reason == "đạt max_steps" and (cfg.max_steps is None or self.state["global_step"] < cfg.max_steps):
                print("➕ Tăng/bỏ giới hạn max_steps: huấn luyện tiếp.")
                self.state["finished"] = False
        if self.state["finished"]:
            print("✔ Run này đã huấn luyện xong trước đó. (Xoá thư mục checkpoint của run nếu muốn huấn luyện lại.)")
            return self.history
        if cfg.max_steps and self.state["global_step"] >= cfg.max_steps:
            print(f"✔ Đã đạt max_steps={cfg.max_steps}.")
            return self.history
        self.model.train()
        t_start = time.time()
        sec_before = self.state["train_seconds"]
        n_bad_loss = n_oom = 0
        stop_reason = None
        loss_acc = nll_acc = tok_acc = 0.0
        since_log = 0

        try:
            for epoch in range(self.state["epoch"], cfg.epochs):
                self.state["epoch"] = epoch
                batches = self.sampler.batches(epoch)
                start_b = self.state["batch_in_epoch"]
                loader = torch.utils.data.DataLoader(self.dataset, batch_sampler=batches[start_b:],
                                                     collate_fn=self.collate_fn, num_workers=2,
                                                     persistent_workers=False, pin_memory=self.device.type == "cuda",
                                                     # generator riêng => DataLoader không "ăn" RNG toàn cục,
                                                     # dropout sau khi resume giống hệt khi chạy liền mạch
                                                     generator=torch.Generator().manual_seed(cfg.seed + epoch))
                pbar = tqdm(total=len(batches), initial=start_b, desc=f"{self.run_name} epoch {epoch + 1}/{cfg.epochs}",
                            leave=False)
                self.optimizer.zero_grad(set_to_none=True)
                micro = 0
                for bi, batch in enumerate(loader, start=start_b):
                    batch = {k: v.to(self.device, non_blocking=True) for k, v in batch.items()}
                    try:
                        with autocast_ctx(self.amp_dtype):
                            loss, nll, ntok = self.loss_fn(self.model, batch, cfg)
                        if not torch.isfinite(loss):
                            n_bad_loss += 1
                            self.optimizer.zero_grad(set_to_none=True)
                            micro = 0
                            if n_bad_loss >= 20:
                                raise TooManyFailures(
                                    "Loss bị NaN/inf 20 lần liên tiếp. Thử: giảm lr, đặt amp='fp32' hoặc 'bf16'.")
                            continue
                        n_bad_loss = 0
                        self.scaler.scale(loss / cfg.grad_accum).backward()
                    except torch.cuda.OutOfMemoryError:
                        n_oom += 1
                        self.optimizer.zero_grad(set_to_none=True)
                        micro = 0
                        del batch
                        torch.cuda.empty_cache()
                        print(f"⚠️  Hết bộ nhớ GPU ở batch {bi} (lần {n_oom}) — bỏ qua batch này.")
                        if n_oom >= 10:
                            raise TooManyFailures("Hết bộ nhớ GPU quá nhiều lần. Giảm max_tokens (vd. chia đôi) "
                                                  "và tăng grad_accum tương ứng.")
                        continue
                    loss_acc += float(loss.detach()) * ntok
                    nll_acc += float(nll) * ntok
                    tok_acc += ntok
                    micro += 1
                    pbar.update(1)
                    self.state["batch_in_epoch"] = bi + 1
                    if micro < cfg.grad_accum:
                        continue
                    micro = 0
                    self.scaler.unscale_(self.optimizer)
                    gnorm = torch.nn.utils.clip_grad_norm_(
                        [p for g in self.optimizer.param_groups for p in g["params"]], cfg.clip_norm)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scheduler.step()
                    self.state["global_step"] += 1
                    since_log += 1
                    gs = self.state["global_step"]

                    if gs % cfg.log_every_steps == 0 and tok_acc > 0:
                        tl = loss_acc / tok_acc
                        ppl = math.exp(min(20, nll_acc / tok_acc))
                        pbar.set_postfix(loss=f"{tl:.3f}", ppl=f"{ppl:.1f}", lr=f"{self.optimizer.param_groups[0]['lr']:.2e}")
                        self._log_history({"step": gs, "epoch": epoch, "train_loss": tl, "train_ppl": ppl,
                                           "grad_norm": float(gnorm), "lr": self.optimizer.param_groups[0]["lr"],
                                           "time": now_str()})
                        loss_acc = nll_acc = tok_acc = 0.0
                    if cfg.eval_every_steps and gs % cfg.eval_every_steps == 0:
                        self.evaluate(tag=f"epoch {epoch + 1}")
                        if self.state["bad_evals"] >= cfg.patience:
                            stop_reason = "early stopping"
                    if gs % cfg.save_every_steps == 0:
                        self.state["train_seconds"] = sec_before + time.time() - t_start
                        self.save_last()
                    if cfg.max_steps and gs >= cfg.max_steps:
                        stop_reason = stop_reason or "đạt max_steps"
                    if cfg.time_budget_hours and (time.time() - t_start) / 3600 > cfg.time_budget_hours:
                        stop_reason = stop_reason or "hết thời gian cho phép (time_budget_hours)"
                    if stop_reason:
                        break
                pbar.close()
                if stop_reason:
                    break
                # cuối epoch
                self.state["epoch"] = epoch + 1
                self.state["batch_in_epoch"] = 0
                if not cfg.eval_every_steps:
                    self.evaluate(tag=f"epoch {epoch + 1}/{cfg.epochs}")
                    if self.state["bad_evals"] >= cfg.patience:
                        stop_reason = "early stopping"
                self.state["train_seconds"] = sec_before + time.time() - t_start
                self.save_last()
                if stop_reason:
                    break
        except KeyboardInterrupt:
            self.state["train_seconds"] = sec_before + time.time() - t_start
            self.save_last()
            print("⏸ Đã dừng tay và lưu checkpoint. Chạy lại ô này để tiếp tục.")
            raise

        self.state["train_seconds"] = sec_before + time.time() - t_start
        # Chỉ coi là "xong hẳn" khi dừng sớm hoặc đủ epoch; dừng vì max_steps/hết giờ thì lần sau chạy tiếp được
        # Đánh giá lần cuối nếu bước hiện tại chưa được đánh giá (vd. dừng vì max_steps giữa epoch)
        if (self.eval_fn is not None and stop_reason != "hết thời gian cho phép (time_budget_hours)"
                and self.state.get("last_eval_step") != self.state["global_step"]):
            self.evaluate(tag="cuối")
        if stop_reason in (None, "early stopping", "đạt max_steps"):
            self.state["finished"] = True
        self.state["stop_reason"] = stop_reason or "đủ số epoch"
        self.save_last()
        save_json({"run": self.run_name, **self.state,
                   "train_time": fmt_seconds(self.state["train_seconds"]), "cfg": self.cfg.to_dict(),
                   **self.extra_meta}, self.ckpt_dir / "train_summary.json")
        print(f"🏁 {self.run_name}: kết thúc ({stop_reason or 'đủ số epoch'}), "
              f"best dev = {self.state['best_score']:.2f} tại step {self.state['best_step']}, "
              f"thời gian train = {fmt_seconds(self.state['train_seconds'])}")
        return self.history

Overwriting envi_lib/train.py


In [9]:
%%writefile envi_lib/metrics.py
"""Các độ đo đánh giá chất lượng dịch máy (phần chạy trong môi trường chính).

  * BLEU, chrF++, TER  : sacreBLEU (chuẩn hoá cách token hoá => điểm so sánh được giữa các bài báo)
  * METEOR             : NLTK (lưu ý: phần đồng nghĩa WordNet chỉ có cho tiếng Anh)
  * Kiểm định ý nghĩa thống kê: paired bootstrap resampling (Koehn, 2004)
  * Phân tích theo độ dài câu

COMET và BERTScore (độ đo dựa trên mạng nơ-ron) chạy trong môi trường Python riêng — xem eval_neural.py.
"""
from __future__ import annotations

import math
from collections import Counter

import numpy as np

MAX_N = 4


def corpus_scores(hyps, refs) -> dict:
    import sacrebleu
    from sacrebleu.metrics import BLEU, CHRF, TER
    assert len(hyps) == len(refs), f"{len(hyps)} bản dịch != {len(refs)} tham chiếu"
    bleu, chrf, ter = BLEU(), CHRF(word_order=2), TER()
    b = bleu.corpus_score(hyps, [refs])
    c = chrf.corpus_score(hyps, [refs])
    t = ter.corpus_score(hyps, [refs])
    return {
        "BLEU": round(b.score, 2), "chrF++": round(c.score, 2), "TER": round(t.score, 2),
        "BP": round(b.bp, 3), "len_ratio": round(b.sys_len / max(1, b.ref_len), 3),
        "p1": round(b.precisions[0], 1), "p2": round(b.precisions[1], 1),
        "p3": round(b.precisions[2], 1), "p4": round(b.precisions[3], 1),
        "_sig_bleu": str(bleu.get_signature()), "_sig_chrf": str(chrf.get_signature()),
        "_sig_ter": str(ter.get_signature()), "_sacrebleu": sacrebleu.__version__,
    }


def quick_bleu(hyps, refs) -> float:
    from sacrebleu.metrics import BLEU
    return BLEU().corpus_score(hyps, [refs]).score


# ----------------------------------------------------------------------------
# BLEU theo thống kê từng câu (để bootstrap nhanh) — kiểm chứng khớp sacreBLEU
# ----------------------------------------------------------------------------

def _tok13a():
    from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a
    return Tokenizer13a()


def _ngrams(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))


def bleu_stats(hyps, refs) -> np.ndarray:
    """Mỗi câu: [len_hyp, len_ref, match_1..4, total_1..4]."""
    tok = _tok13a()
    out = np.zeros((len(hyps), 2 + 2 * MAX_N), dtype=np.int64)
    for i, (h, r) in enumerate(zip(hyps, refs)):
        ht, rt = tok(h.strip()).split(), tok(r.strip()).split()
        out[i, 0], out[i, 1] = len(ht), len(rt)
        for n in range(1, MAX_N + 1):
            hc, rc = _ngrams(ht, n), _ngrams(rt, n)
            out[i, 1 + n] = sum(min(c, rc[g]) for g, c in hc.items())
            out[i, 1 + MAX_N + n] = max(0, len(ht) - n + 1)
    return out


def bleu_from_stats(s) -> float:
    """Giống sacreBLEU mặc định (smooth='exp')."""
    sys_len, ref_len = s[0], s[1]
    if sys_len == 0:
        return 0.0
    log_sum, smooth = 0.0, 1.0
    for n in range(MAX_N):
        m, t = s[2 + n], s[2 + MAX_N + n]
        if t == 0:
            return 0.0
        if m == 0:
            smooth *= 2
            p = 1.0 / (smooth * t)
        else:
            p = m / t
        log_sum += math.log(p)
    bp = 1.0 if sys_len > ref_len else math.exp(1 - ref_len / sys_len)
    return 100 * bp * math.exp(log_sum / MAX_N)


def paired_bootstrap(stats_base, stats_sys, n_samples=1000, seed=12345):
    """So sánh hệ thống 'sys' với 'base' trên cùng tập test.

    Trả về: delta BLEU, khoảng tin cậy 95% của mỗi hệ thống, p-value (xác suất sys KHÔNG tốt hơn base).
    """
    rng = np.random.RandomState(seed)
    N = len(stats_base)
    bb, bs = [], []
    for _ in range(n_samples):
        idx = rng.randint(0, N, N)
        bb.append(bleu_from_stats(stats_base[idx].sum(0)))
        bs.append(bleu_from_stats(stats_sys[idx].sum(0)))
    bb, bs = np.array(bb), np.array(bs)
    delta = bs - bb
    return {
        "base_bleu": round(bleu_from_stats(stats_base.sum(0)), 2),
        "sys_bleu": round(bleu_from_stats(stats_sys.sum(0)), 2),
        "delta": round(float(delta.mean()), 2),
        "base_ci95": (round(float(np.percentile(bb, 2.5)), 2), round(float(np.percentile(bb, 97.5)), 2)),
        "sys_ci95": (round(float(np.percentile(bs, 2.5)), 2), round(float(np.percentile(bs, 97.5)), 2)),
        "p_value": round(float((delta <= 0).mean()), 4),
    }


def bootstrap_mean_ci(values, n_samples=1000, seed=12345):
    rng = np.random.RandomState(seed)
    v = np.asarray(values, dtype=float)
    means = [v[rng.randint(0, len(v), len(v))].mean() for _ in range(n_samples)]
    return float(v.mean()), (float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)))


def paired_bootstrap_mean(base_vals, sys_vals, n_samples=1000, seed=12345):
    rng = np.random.RandomState(seed)
    a, b = np.asarray(base_vals, float), np.asarray(sys_vals, float)
    d = []
    for _ in range(n_samples):
        idx = rng.randint(0, len(a), len(a))
        d.append(b[idx].mean() - a[idx].mean())
    d = np.array(d)
    return {"delta": float(b.mean() - a.mean()), "p_value": float((d <= 0).mean())}


# ----------------------------------------------------------------------------
# METEOR
# ----------------------------------------------------------------------------

_METEOR_OK = None


def _meteor_ready() -> bool:
    """Tải WordNet MỘT lần; nếu không được thì báo một lần và bỏ qua METEOR."""
    global _METEOR_OK
    if _METEOR_OK is None:
        # Lưu ý: nltk.download('wordnet') chỉ tải file .zip (không giải nén) => nltk.data.find('corpora/wordnet')
        # báo không thấy dù vẫn dùng được.  Vì vậy kiểm tra bằng cách DÙNG THỬ WordNet.
        def usable():
            from nltk.corpus import wordnet
            wordnet.ensure_loaded()
            return len(wordnet.synsets("dog")) > 0
        try:
            import nltk
            try:
                ok = usable()
            except LookupError:
                for res in ("wordnet", "omw-1.4"):
                    nltk.download(res, quiet=True)
                ok = usable()
            _METEOR_OK = bool(ok)
        except Exception as e:  # noqa: BLE001
            print(f"⚠️  Không tải được WordNet cho METEOR — bỏ qua METEOR ({type(e).__name__}).")
            _METEOR_OK = False
    return _METEOR_OK


def meteor(hyps, refs):
    """METEOR trung bình theo câu (NLTK).  Trả về (None, None) nếu không có WordNet."""
    if not _meteor_ready():
        return None, None
    from nltk.translate.meteor_score import meteor_score
    scores = [meteor_score([r.lower().split()], h.lower().split()) for h, r in zip(hyps, refs)]
    return round(100 * float(np.mean(scores)), 2), scores


# ----------------------------------------------------------------------------
# Phân tích theo độ dài
# ----------------------------------------------------------------------------

LENGTH_BINS = [(1, 10), (11, 20), (21, 30), (31, 50), (51, 10_000)]


def bleu_by_length(hyps, refs, srcs, bins=LENGTH_BINS):
    """BLEU theo nhóm độ dài câu NGUỒN (số từ tiếng Anh)."""
    from sacrebleu.metrics import BLEU
    out = {}
    lens = [len(s.split()) for s in srcs]
    for lo, hi in bins:
        idx = [i for i, L in enumerate(lens) if lo <= L <= hi]
        label = f"{lo}-{hi}" if hi < 10_000 else f">{lo - 1}"
        if not idx:
            continue
        out[label] = {"n": len(idx), "BLEU": round(BLEU().corpus_score([hyps[i] for i in idx],
                                                                       [[refs[i] for i in idx]]).score, 2)}
    return out

Overwriting envi_lib/metrics.py


In [10]:
%%writefile envi_lib/hf_utils.py
"""Làm việc với mô hình pretrained của Hugging Face — tương thích transformers 4.x và 5.x.

Những điểm khác biệt giữa các phiên bản đã được xử lý tập trung ở đây:
  * tham số kiểu số thực khi tải mô hình: `dtype` (mới) / `torch_dtype` (cũ)
  * `batch_decode` gộp vào `decode` ở v5
  * v5 mặc định use_cache=False => truyền use_cache=True khi generate để sinh nhanh
  * pipeline 'translation' bị bỏ ở v5 => không dùng pipeline
"""
from __future__ import annotations

import json
import os
import re

import torch

# DataLoader dùng tiến trình con (fork); tắt song song của tokenizers để tránh cảnh báo/treo
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def patch_peft_old_torchao():
    """Colab cài sẵn torchao bản cũ (0.10); peft >= 0.18 thấy torchao < 0.16 là báo ImportError ngay khi tạo LoRA,
    dù đồ án KHÔNG dùng torchao (chỉ dùng cho lượng tử hoá).  Ta báo cho peft là "không có torchao".
    Không gỡ/cài lại gói nào => không ảnh hưởng môi trường Colab."""
    import importlib
    import importlib.util
    import sys
    if importlib.util.find_spec("torchao") is None or importlib.util.find_spec("peft") is None:
        return False
    try:
        from importlib.metadata import version
        from packaging.version import Version
        if Version(version("torchao")) >= Version("0.16.0"):
            return False
    except Exception:  # noqa: BLE001
        return False
    patched = False
    for name in ("peft.import_utils", "peft.tuners.lora.torchao", "peft.utils.quantization_utils"):
        try:
            mod = sys.modules.get(name) or importlib.import_module(name)
        except Exception:  # noqa: BLE001
            continue
        if hasattr(mod, "is_torchao_available"):
            mod.is_torchao_available = lambda: False
            patched = True
    if patched:
        print("ℹ️  Phát hiện torchao bản cũ của Colab — đã báo peft bỏ qua torchao (không dùng tới trong đồ án).")
    return patched


patch_peft_old_torchao()

from .common import autocast_ctx, normalize_text
from .data import sorted_eval_batches


def tf_version():
    import transformers
    from packaging.version import Version
    return Version(transformers.__version__)


def dtype_kw(dtype):
    """Tham số kiểu số thực cho from_pretrained theo phiên bản transformers."""
    from packaging.version import Version
    return {"dtype": dtype} if tf_version() >= Version("4.56.0") else {"torch_dtype": dtype}


def resolve_model_id(model_id: str) -> str:
    """Cho phép thay đường dẫn mô hình (dùng khi kiểm thử offline): ENVI_MODEL_OVERRIDES='{"id": "path"}'."""
    ov = os.environ.get("ENVI_MODEL_OVERRIDES")
    if ov:
        return json.loads(ov).get(model_id, model_id)
    return model_id


def decode_batch(tok, ids):
    if torch.is_tensor(ids):
        ids = ids.tolist()
    if hasattr(tok, "batch_decode"):
        return tok.batch_decode(ids, skip_special_tokens=True)
    out = tok.decode(ids, skip_special_tokens=True)
    return out if isinstance(out, list) else [out]


# ----------------------------------------------------------------------------
# Mô hình Encoder-Decoder pretrained (Marian, T5/envit5, mT5, NLLB)
# ----------------------------------------------------------------------------

class Seq2SeqSpec:
    """Mô tả cách dùng từng họ mô hình: tiền tố đầu vào, ngôn ngữ đích, hậu xử lý đầu ra."""

    def __init__(self, family: str, src_prefix: str = "", tgt_prefix: str = "",
                 src_lang: str | None = None, tgt_lang: str | None = None, fp16_ok: bool = True):
        self.family = family
        self.src_prefix = src_prefix
        self.tgt_prefix = tgt_prefix          # tiền tố trong NHÃN khi fine-tune (envit5: "vi: ")
        self.src_lang = src_lang              # NLLB: eng_Latn
        self.tgt_lang = tgt_lang              # NLLB: vie_Latn
        self.fp16_ok = fp16_ok                # họ T5 bị tràn số (NaN) với fp16

    def make_src(self, s):
        return self.src_prefix + s

    def make_tgt(self, t):
        return self.tgt_prefix + t

    def postprocess(self, out: str) -> str:
        out = normalize_text(out)
        if self.tgt_prefix and out.lower().startswith(self.tgt_prefix.strip().lower()):
            out = out[len(self.tgt_prefix.strip()):].lstrip(" :")
        return normalize_text(out)


# File cần để dựng lại tokenizer từ mô hình SentencePiece gốc (bỏ qua tokenizer.json)
_TOK_FILES = ["tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "config.json",
              "*.model", "*.spm", "vocab*.json", "vocab*.txt", "merges*.txt"]


def _tokenizer_from_sentencepiece(path, **tok_kw):
    """Dựng tokenizer KHÔNG dùng tokenizer.json, chỉ dùng file SentencePiece gốc (spiece.model ...).

    Dùng khi tokenizer.json của mô hình được tạo bằng phiên bản cũ, không đọc được bằng transformers 5.x
    (vd. VietAI/envit5-translation: lỗi "'dict' object is not an instance of 'Sequence'").
    """
    import shutil
    import tempfile
    from pathlib import Path
    from transformers import AutoTokenizer
    if os.path.isdir(path):
        src = Path(path)
    else:
        from huggingface_hub import snapshot_download
        src = Path(snapshot_download(path, allow_patterns=_TOK_FILES))
    if not any(src.glob("*.model")) and not any(src.glob("*.spm")):
        raise FileNotFoundError(f"{path}: không có file SentencePiece (*.model) để dựng lại tokenizer")
    try:
        import google.protobuf  # noqa: F401  (transformers cần protobuf để đọc file .model)
    except ImportError as e:
        raise ImportError("Cần gói protobuf để đọc spiece.model: chạy `%pip install protobuf` rồi "
                          "Runtime → Restart session, sau đó chạy lại notebook.") from e
    dst = Path(tempfile.mkdtemp(prefix="tok_"))
    for f in src.iterdir():
        if f.is_file() and f.name != "tokenizer.json":
            shutil.copy(f, dst / f.name)
    return AutoTokenizer.from_pretrained(str(dst), **tok_kw)


def load_tokenizer(path, **tok_kw):
    from transformers import AutoTokenizer
    try:
        return AutoTokenizer.from_pretrained(path, **tok_kw)
    except (TypeError, ValueError, KeyError, AttributeError, OSError) as e:
        if isinstance(e, OSError) and "tokenizer" not in str(e).lower():
            raise   # lỗi mạng / không tìm thấy mô hình: báo ngay, không che giấu
        print(f"⚠️  Không đọc được tokenizer.json của {path} ({type(e).__name__}: {str(e)[:120]}).\n"
              "   → Dựng lại tokenizer từ file SentencePiece gốc.")
        return _tokenizer_from_sentencepiece(path, **tok_kw)


def check_tokenizer(tok, model,
                    sample="Tôi là giáo viên Tin học ở Thành phố Hồ Chí Minh. I teach C++ to students."):
    """Kiểm tra tokenizer có khớp mô hình không.  Trả về (lỗi_nặng, số_token_<unk>_trên_câu_mẫu).

    Lỗi nặng: id <pad>/</s> khác mô hình; số token vượt xa bảng embedding.
    """
    hard = []
    V = model.get_input_embeddings().weight.shape[0]
    if len(tok) > 1.01 * V:
        hard.append(f"tokenizer có {len(tok)} token nhưng bảng embedding chỉ có {V}")
    cfg = model.config
    for name in ("pad_token_id", "eos_token_id"):
        a, b = getattr(tok, name, None), getattr(cfg, name, None)
        if a is not None and b is not None and not isinstance(b, list) and a != b:
            hard.append(f"{name}: tokenizer = {a}, mô hình = {b}")
    ids = tok(sample, add_special_tokens=False)["input_ids"]
    n_unk = ids.count(tok.unk_token_id) if tok.unk_token_id is not None else 0
    if len(ids) < len(sample.split()) // 2:
        hard.append(f"câu mẫu {len(sample.split())} từ chỉ được tách thành {len(ids)} token")
    return hard, n_unk


def load_seq2seq(model_id, spec: Seq2SeqSpec, device, for_training=False):
    from transformers import AutoModelForSeq2SeqLM
    path = resolve_model_id(model_id)
    tok_kw = {"src_lang": spec.src_lang, "tgt_lang": spec.tgt_lang} if spec.src_lang else {}
    tok = load_tokenizer(path, **tok_kw)
    model = AutoModelForSeq2SeqLM.from_pretrained(path, **dtype_kw(torch.float32)).to(device)
    hard, n_unk = check_tokenizer(tok, model)
    if hard or n_unk > 2:
        # tokenizer.json đọc được nhưng có dấu hiệu KHÔNG khớp mô hình => thử dựng lại từ SentencePiece gốc
        print(f"⚠️  Tokenizer nghi không khớp mô hình ({'; '.join(hard) or f'{n_unk} token <unk> trên câu mẫu'}).")
        try:
            tok2 = _tokenizer_from_sentencepiece(path, **tok_kw)
            hard2, n_unk2 = check_tokenizer(tok2, model)
            if not hard2 and (hard or n_unk2 < n_unk):
                print("   → Dùng tokenizer dựng lại từ file SentencePiece gốc.")
                tok, hard, n_unk = tok2, hard2, n_unk2
        except (FileNotFoundError, OSError, ValueError, TypeError) as e:
            print(f"   (không dựng lại được từ SentencePiece: {str(e)[:100]})")
    if hard:
        raise RuntimeError("Tokenizer không khớp mô hình: " + "; ".join(hard))
    if n_unk > 2:
        print(f"⚠️  Tokenizer sinh {n_unk} token <unk> trên câu mẫu — kết quả dịch có thể bị ảnh hưởng.")
    preflight_seq2seq(tok, model, spec)
    model.train(for_training)
    return tok, model


def preflight_seq2seq(tok, model, spec: Seq2SeqSpec):
    """Kiểm tra các yêu cầu đặc thù TRƯỚC khi chạy lâu."""
    vocab = tok.get_vocab()
    if spec.family == "marian":
        m = re.match(r"\s*(>>\w+<<)", spec.src_prefix)
        if m and m.group(1) not in vocab:
            raise RuntimeError(f"Tokenizer Marian không có token {m.group(1)} — kiểm tra lại mã ngôn ngữ đích.")
    if spec.tgt_lang:
        tid = tok.convert_tokens_to_ids(spec.tgt_lang)
        if tid is None or tid == tok.unk_token_id:
            raise RuntimeError(f"Không tìm thấy mã ngôn ngữ {spec.tgt_lang} trong tokenizer.")
    if model.config.decoder_start_token_id is None and getattr(model.generation_config, "decoder_start_token_id", None) is None:
        raise RuntimeError("Mô hình thiếu decoder_start_token_id.")


def seq2seq_gen_kwargs(tok, spec: Seq2SeqSpec, beam_size, max_new_tokens):
    # Encoder-decoder: max_length tính theo số token phía decoder (+1 token khởi đầu). Dùng max_length thay vì
    # max_new_tokens để ghi đè hẳn giá trị mặc định trong generation_config (Marian đặt sẵn 512) mà không bị cảnh báo.
    kw = dict(num_beams=beam_size, max_length=max_new_tokens + 1, do_sample=False, use_cache=True,
              length_penalty=1.0, early_stopping=beam_size > 1)
    if beam_size == 1:
        kw.pop("early_stopping")
    if spec.tgt_lang:
        kw["forced_bos_token_id"] = tok.convert_tokens_to_ids(spec.tgt_lang)
    return kw


@torch.no_grad()
def hf_seq2seq_translate(model, tok, spec: Seq2SeqSpec, srcs, beam_size=4, batch_size=32,
                         max_src_tokens=256, amp_dtype=None, progress=True):
    from tqdm.auto import tqdm
    model.eval()
    device = next(model.parameters()).device
    texts = [spec.make_src(s) for s in srcs]
    lens = [len(t) for t in texts]
    bs = max(1, batch_size // max(1, beam_size // 2))
    batches = sorted_eval_batches(lens, bs)
    out = [None] * len(texts)
    use_amp = amp_dtype if (amp_dtype != torch.float16 or spec.fp16_ok) else None
    it = tqdm(batches, desc=f"Dịch (beam={beam_size})", leave=False) if progress else batches
    for idx in it:
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, truncation=True,
                  max_length=max_src_tokens)
        enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}  # bỏ token_type_ids nếu có
        max_new = int(enc["input_ids"].size(1) * 1.6 + 10)
        with autocast_ctx(use_amp):
            gen = model.generate(**enc, **seq2seq_gen_kwargs(tok, spec, beam_size, max_new))
        for i, h in zip(idx, decode_batch(tok, gen)):
            out[i] = spec.postprocess(h)
    return out


def make_hf_seq2seq_features(tok, spec: Seq2SeqSpec, srcs, tgts, max_len=256):
    """Token hoá cặp câu cho fine-tune.  Trả về list (input_ids, labels) và số cặp bị cắt."""
    enc = tok([spec.make_src(s) for s in srcs], text_target=[spec.make_tgt(t) for t in tgts],
              truncation=True, max_length=max_len)
    feats = list(zip(enc["input_ids"], enc["labels"]))
    n_trunc = sum(1 for a, b in feats if len(a) >= max_len or len(b) >= max_len)
    return feats, n_trunc


def hf_seq2seq_collate(batch, pad_id):
    """batch: list (idx, input_ids, labels)."""
    from .data import IGNORE, _pad
    ids = _pad([b[1] for b in batch], pad_id)
    lab = _pad([b[2] for b in batch], IGNORE)
    return {"input_ids": ids, "attention_mask": (ids != pad_id) if pad_id is not None else torch.ones_like(ids, dtype=torch.bool),
            "labels": lab}


def hf_seq2seq_loss(model, batch, cfg):
    """Tự tạo decoder_input_ids từ nhãn và tự tính loss (có label smoothing).

    Không truyền `labels` vào mô hình để tránh HF tính thêm một lần cross-entropy trên toàn bộ
    từ vựng (mT5/NLLB có 250k từ => tốn thêm vài GB bộ nhớ GPU).
    """
    from .models import label_smoothed_ce
    labels = batch["labels"]
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    if hasattr(base, "prepare_decoder_input_ids_from_labels"):
        dec_in = base.prepare_decoder_input_ids_from_labels(labels=labels)
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"].long(),
                    decoder_input_ids=dec_in)
    else:
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"].long(), labels=labels)
    return label_smoothed_ce(out.logits, labels, cfg.label_smoothing)


class FeatDataset(torch.utils.data.Dataset):
    def __init__(self, feats):
        self.feats = feats

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, i):
        a, b = self.feats[i]
        return i, a, b


# ----------------------------------------------------------------------------
# LLM decoder-only (Qwen ...) — prompt, LoRA
# ----------------------------------------------------------------------------

SYSTEM_PROMPT = "You are a professional English-Vietnamese translator."
USER_TEMPLATE = ("Translate the following English sentence into Vietnamese. "
                 "Output only the Vietnamese translation, nothing else.\n\n{src}")


def build_messages(src, shots=()):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for s, t in shots:
        msgs.append({"role": "user", "content": USER_TEMPLATE.format(src=s)})
        msgs.append({"role": "assistant", "content": t})
    msgs.append({"role": "user", "content": USER_TEMPLATE.format(src=src)})
    return msgs


def chat_prompt(tok, src, shots=()):
    kw = {}
    if "enable_thinking" in (getattr(tok, "chat_template", None) or ""):
        kw["enable_thinking"] = False   # Qwen3: tắt chế độ "suy nghĩ"
    s = tok.apply_chat_template(build_messages(src, shots), tokenize=False, add_generation_prompt=True, **kw)
    if not isinstance(s, str):
        raise RuntimeError("apply_chat_template(tokenize=False) không trả về chuỗi.")
    return s


def chat_train_texts(tok, src, tgt):
    """(prompt_text, target_text) sao cho prompt + target = hội thoại đầy đủ theo chat template."""
    prompt = chat_prompt(tok, src)
    full = tok.apply_chat_template(build_messages(src) + [{"role": "assistant", "content": tgt}],
                                   tokenize=False, add_generation_prompt=False)
    if not full.startswith(prompt):
        raise RuntimeError("Chat template không nhất quán giữa prompt và hội thoại đầy đủ "
                           "(thường gặp ở mô hình có chế độ suy nghĩ). Hãy dùng Qwen2.5-Instruct.")
    target = full[len(prompt):]
    eos = getattr(tok, "eos_token", None)
    if eos and eos in target:   # bỏ phần sau token kết thúc (vd. "\n" của ChatML) — không cần học
        target = target[: target.index(eos) + len(eos)]
    return prompt, target


_PREFIX_RE = re.compile(r"^(vietnamese( translation)?|translation|bản dịch|dịch|tiếng việt)\s*[:：]\s*", re.I)


def clean_llm_output(text: str) -> str:
    """Hậu xử lý đầu ra LLM: lấy dòng đầu không rỗng, bỏ tiền tố 'Translation:' và dấu ngoặc kép bao ngoài."""
    text = (text or "").replace("<think>", "").replace("</think>", "")
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    s = lines[0] if lines else ""
    s = _PREFIX_RE.sub("", s)
    if len(s) >= 2 and s[0] == s[-1] and s[0] in "\"'“”":
        s = s[1:-1]
    if len(s) >= 2 and s[0] == "“" and s[-1] == "”":
        s = s[1:-1]
    return normalize_text(s)


def load_llm(model_id, device, dtype):
    from transformers import AutoModelForCausalLM
    path = resolve_model_id(model_id)
    tok = load_tokenizer(path)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if not getattr(tok, "chat_template", None):
        raise RuntimeError(f"{model_id} không có chat template — hãy dùng bản *-Instruct.")
    model = AutoModelForCausalLM.from_pretrained(path, **dtype_kw(dtype)).to(device)
    if model.generation_config.pad_token_id is None:
        model.generation_config.pad_token_id = tok.pad_token_id
    return tok, model


def make_llm_features(tok, srcs, tgts, max_len=384):
    feats, n_skip = [], 0
    for s, t in zip(srcs, tgts):
        p, tgt = chat_train_texts(tok, s, t)
        p_ids = tok(p, add_special_tokens=False)["input_ids"]
        t_ids = tok(tgt, add_special_tokens=False)["input_ids"]
        if len(p_ids) + len(t_ids) > max_len:
            n_skip += 1
            continue
        feats.append((p_ids, t_ids))
    return feats, n_skip


def llm_collate(batch, pad_id):
    from .data import IGNORE
    seqs = [list(p) + list(t) for _, p, t in batch]
    labs = [[IGNORE] * len(p) + list(t) for _, p, t in batch]
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    lab = torch.full((len(seqs), L), IGNORE, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        ids[i, :len(s)] = torch.tensor(s)
        lab[i, :len(l)] = torch.tensor(l)
        mask[i, :len(s)] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": mask}


def llm_loss(model, batch, cfg):
    from .models import label_smoothed_ce
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1]
    labels = batch["labels"][:, 1:]
    return label_smoothed_ce(logits, labels, cfg.label_smoothing)


@torch.no_grad()
def llm_translate(model, tok, srcs, shots=(), batch_size=32, max_new_tokens=256, amp_dtype=None,
                  progress=True, return_raw=False):
    from tqdm.auto import tqdm
    model.eval()
    device = next(model.parameters()).device
    old_side = tok.padding_side
    tok.padding_side = "left"   # sinh văn bản theo batch với decoder-only: BẮT BUỘC đệm trái
    prompts = [chat_prompt(tok, s, shots) for s in srcs]
    batches = sorted_eval_batches([len(p) for p in prompts], batch_size)
    out, raw = [None] * len(srcs), [None] * len(srcs)
    eos = model.generation_config.eos_token_id
    it = tqdm(batches, desc="LLM dịch", leave=False) if progress else batches
    try:
        for idx in it:
            enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False)
            enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}
            with autocast_ctx(amp_dtype):
                gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                                     use_cache=True, pad_token_id=tok.pad_token_id, eos_token_id=eos)
            new = gen[:, enc["input_ids"].size(1):]
            for i, h in zip(idx, decode_batch(tok, new)):
                raw[i] = h
                out[i] = clean_llm_output(h)
    finally:
        tok.padding_side = old_side
    return (out, raw) if return_raw else out


def enable_grad_ckpt(peft_model):
    """Gradient checkpointing: giảm bộ nhớ kích hoạt (activation) ~3-5 lần, đổi lại chậm hơn ~30%."""
    base = peft_model.get_base_model() if hasattr(peft_model, "get_base_model") else peft_model
    try:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        base.gradient_checkpointing_enable()
    if hasattr(base, "enable_input_require_grads"):
        base.enable_input_require_grads()
    base.config.use_cache = False

Overwriting envi_lib/hf_utils.py


In [11]:
%%writefile envi_lib/neural_env.py
"""Tạo môi trường Python RIÊNG cho COMET + BERTScore và chạy eval_neural.py trong đó.

Vì sao: `unbabel-comet` đòi numpy<2, transformers<5, huggingface-hub<1 — cài thẳng vào Colab sẽ
hạ cấp các thư viện đang dùng và làm hỏng notebook.  Môi trường riêng (venv) giải quyết triệt để:
hai bộ thư viện không đụng nhau.

PyTorch trong venv được cài CÙNG phiên bản + CÙNG bản CUDA với PyTorch của Colab
=> chắc chắn tương thích driver GPU của máy.
"""
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

# setuptools<81: torchmetrics 0.10 (phụ thuộc của COMET) còn dùng pkg_resources, đã bị gỡ khỏi setuptools mới
EVAL_PKGS = ["unbabel-comet==2.2.7", "bert-score==0.3.13", "setuptools<81"]


def _run(cmd, log=print):
    log("$ " + " ".join(map(str, cmd)))
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        line = line.rstrip()
        tail = (tail + [line])[-40:]
        if any(k in line for k in ("COMET", "BERTScore", "XONG", "!!", "Error", "error:", "Traceback")):
            log("   " + line)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}):\n" + "\n".join(tail))


def ensure_eval_env(env_dir="/content/eval_env", python: str | None = None, log=print) -> Path:
    env_dir = Path(env_dir)
    py = env_dir / "bin" / "python"
    marker = env_dir / ".ready"
    if marker.exists() and py.exists():
        log(f"✔ Môi trường đánh giá đã sẵn sàng: {env_dir}")
        return py
    import torch
    _run([sys.executable, "-m", "pip", "install", "-q", "uv"], log)
    _run([sys.executable, "-m", "uv", "venv", "--clear", env_dir, "--python", python or sys.executable], log)
    base, _, variant = torch.__version__.partition("+")
    installed = False
    if variant.startswith("cu") or variant == "cpu":
        try:
            _run([sys.executable, "-m", "uv", "pip", "install", "--python", py, f"torch=={base}",
                  "--index-url", f"https://download.pytorch.org/whl/{variant}"], log)
            installed = True
        except RuntimeError as e:
            log(f"⚠️  Không cài được torch từ download.pytorch.org ({str(e)[-200:]}); thử PyPI.")
    if not installed:
        _run([sys.executable, "-m", "uv", "pip", "install", "--python", py, f"torch=={base}"], log)
    _run([sys.executable, "-m", "uv", "pip", "install", "--python", py, *EVAL_PKGS], log)
    _run([py, "-c", "import comet, bert_score, torch, transformers, numpy; "
                    "print('OK torch', torch.__version__, 'cuda', torch.cuda.is_available(), "
                    "'transformers', transformers.__version__, 'numpy', numpy.__version__)"], log)
    marker.write_text("ok")
    log(f"✔ Đã tạo môi trường đánh giá: {env_dir}")
    return py


def run_neural_metrics(py, script, src_file, ref_file, hyp_files, out_json, comet=True, bertscore=True,
                       batch=64, bertscore_model="xlm-roberta-large", extra_args=(), log=print):
    cmd = [py, script, "--src", src_file, "--ref", ref_file, "--out", out_json, "--batch", str(batch),
           "--bertscore-model", bertscore_model, "--hyp", *hyp_files, *extra_args]
    if not comet:
        cmd.append("--no-comet")
    if not bertscore:
        cmd.append("--no-bertscore")
    _run(cmd, log)

Overwriting envi_lib/neural_env.py


In [12]:
%%writefile envi_lib/eval_neural.py
"""Tính COMET và BERTScore — chạy trong MÔI TRƯỜNG PYTHON RIÊNG (eval_env).

Lý do tách riêng: gói `unbabel-comet` yêu cầu numpy<2 và transformers<5; cài vào môi trường chính
của Colab sẽ hạ cấp thư viện và làm hỏng các notebook khác.

Script này KHÔNG import envi_lib, chỉ dùng thư viện chuẩn + comet + bert_score.
Kết quả được lưu đệm theo mã băm nội dung file => chạy lại chỉ tính hệ thống mới/thay đổi.

Cách dùng:
  python eval_neural.py --src test.en --ref test.vi --hyp a.vi.txt b.vi.txt --out neural.json [--no-comet] [--no-bertscore]
"""
import argparse
import hashlib
import json
import os
import sys
import unicodedata


def read(path):
    with open(path, encoding="utf-8") as f:
        return [unicodedata.normalize("NFC", l.rstrip("\n")) for l in f]


def sha1(lines):
    return hashlib.sha1("\n".join(lines).encode("utf-8")).hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--ref", required=True)
    ap.add_argument("--hyp", nargs="+", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--comet-model", default="Unbabel/wmt22-comet-da")
    ap.add_argument("--bertscore-model", default="xlm-roberta-large")
    ap.add_argument("--bertscore-layers", type=int, default=None, help="chỉ cần khi dùng mô hình ngoài danh sách của bert_score")
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--no-comet", action="store_true")
    ap.add_argument("--no-bertscore", action="store_true")
    a = ap.parse_args()

    import torch
    src, ref = read(a.src), read(a.ref)
    assert len(src) == len(ref), "src/ref lệch số dòng"
    results = {}
    if os.path.exists(a.out):
        with open(a.out, encoding="utf-8") as f:
            results = json.load(f)

    systems = {}
    for p in a.hyp:
        name = os.path.basename(p).replace(".vi.txt", "")
        hyp = read(p)
        if len(hyp) != len(ref):
            print(f"!! Bỏ qua {name}: {len(hyp)} dòng != {len(ref)}", flush=True)
            continue
        systems[name] = hyp

    def save():
        tmp = a.out + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False)
        os.replace(tmp, a.out)

    gpus = 1 if torch.cuda.is_available() else 0

    if not a.no_comet:
        todo = {n: h for n, h in systems.items()
                if results.get(n, {}).get("comet_hash") != sha1(h) + a.comet_model}
        if todo:
            from comet import download_model, load_from_checkpoint
            print(f"COMET ({a.comet_model}) cho {len(todo)} hệ thống ...", flush=True)
            model = load_from_checkpoint(download_model(a.comet_model))
            for n, h in todo.items():
                data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src, h, ref)]
                o = model.predict(data, batch_size=a.batch, gpus=gpus, progress_bar=False)
                r = results.setdefault(n, {})
                r["COMET"] = round(100 * float(o.system_score), 2)
                r["COMET_seg"] = [round(float(x), 4) for x in o.scores]
                r["comet_model"] = a.comet_model
                r["comet_hash"] = sha1(h) + a.comet_model
                print(f"  {n}: COMET = {r['COMET']}", flush=True)
                save()
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if not a.no_bertscore:
        todo = {n: h for n, h in systems.items()
                if results.get(n, {}).get("bs_hash") != sha1(h) + a.bertscore_model}
        if todo:
            from bert_score import BERTScorer
            print(f"BERTScore ({a.bertscore_model}) cho {len(todo)} hệ thống ...", flush=True)
            scorer = BERTScorer(model_type=a.bertscore_model, num_layers=a.bertscore_layers, lang="vi",
                                batch_size=a.batch, device="cuda" if gpus else "cpu")
            for n, h in todo.items():
                # câu rỗng làm bert_score lỗi ở một số phiên bản => thay bằng "." (điểm sẽ thấp, đúng bản chất)
                hh = [x if x.strip() else "." for x in h]
                P, R, F = scorer.score(hh, ref)
                r = results.setdefault(n, {})
                r["BERTScore_P"] = round(100 * float(P.mean()), 2)
                r["BERTScore_R"] = round(100 * float(R.mean()), 2)
                r["BERTScore_F1"] = round(100 * float(F.mean()), 2)
                r["BERTScore_seg"] = [round(float(x), 4) for x in F.tolist()]
                r["bertscore_model"] = a.bertscore_model
                r["bs_hash"] = sha1(h) + a.bertscore_model
                print(f"  {n}: BERTScore F1 = {r['BERTScore_F1']}", flush=True)
                save()
    save()
    print("XONG", flush=True)


if __name__ == "__main__":
    sys.exit(main())

Overwriting envi_lib/eval_neural.py


In [13]:
%%writefile envi_lib/selftest.py
"""Bộ kiểm thử nhanh — chạy trong notebook 00 (và trước mỗi lần train) để phát hiện lỗi SỚM.

Mỗi kiểm thử in ✅ hoặc ❌ kèm giải thích.  Toàn bộ chạy < 2 phút trên CPU, vài giây trên GPU.
"""
from __future__ import annotations

import math
import random
import tempfile
import traceback
from functools import partial
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from . import data as D
from . import decoding as DEC
from . import metrics as M
from . import models as MD
from .common import normalize_text
from .train import TrainConfig, Trainer

V = 40  # từ vựng giả lập cho kiểm thử


def _tiny_s2s(**kw):
    cfg = MD.Seq2SeqConfig(vocab_size=V, d_model=32, n_heads=4, d_ff=64, enc_layers=2, dec_layers=2,
                           dropout=0.0, attn_dropout=0.0, max_len=128, **kw)
    return MD.TransformerSeq2Seq(cfg)


def _tiny_gpt(**kw):
    cfg = MD.GPTConfig(vocab_size=V, d_model=32, n_heads=4, d_ff=64, n_layers=3, dropout=0.0,
                       attn_dropout=0.0, max_len=128, **kw)
    return MD.GPTDecoder(cfg)


SEP = 4  # id <sep> giả lập


# ---------------------------------------------------------------------------- kiểm thử
def t_normalize():
    a = "Việt Nam  ​ đẹp"   # dạng tổ hợp + khoảng trắng lạ
    assert normalize_text(a) == "Việt Nam đẹp", normalize_text(a)


def t_clean_line():
    assert D.clean_line("It &apos;s a &quot; test &quot; .", "en") == 'It\'s a "test".'
    assert D.clean_line("Tôi nói , &quot; Xin chào &quot; .", "vi") == 'Tôi nói, "Xin chào".'
    assert D.clean_line("   ", "vi") == ""


def t_sampler():
    lengths = np.random.RandomState(0).randint(1, 60, 500)
    s = D.TokenBatchSampler(lengths, max_tokens=300, seed=1)
    b0, b0b, b1 = s.batches(0), s.batches(0), s.batches(1)
    assert b0 == b0b, "batch phải xác định theo epoch (để resume đúng)"
    assert b0 != b1
    flat = sorted(i for b in b0 for i in b)
    assert flat == list(range(500)), "mỗi câu xuất hiện đúng 1 lần"
    for b in b0:
        assert len(b) == 1 or max(lengths[b]) * len(b) <= 300


def t_collate():
    batch = [(0, [5, 6, 7], [8, 9]), (1, [5], [8, 9, 10, 11])]
    g = D.gpt_collate(batch, sep=SEP)
    # câu 0: <s> 5 6 7 <sep> 8 9 </s>  -> nhãn chỉ ở 8 9 </s>
    lab = g["labels"][0].tolist()
    assert lab[:4] == [-100] * 4 and lab[4:7] == [8, 9, D.EOS_ID], lab
    s = D.seq2seq_collate(batch)
    assert s["src"][0].tolist() == [5, 6, 7, D.EOS_ID]
    assert s["tgt_in"][1].tolist() == [D.BOS_ID, 8, 9, 10, 11]
    assert s["tgt_out"][1].tolist() == [8, 9, 10, 11, D.EOS_ID]


def t_attention_sdpa_vs_manual():
    torch.manual_seed(0)
    m = _tiny_s2s().eval()
    src = torch.randint(5, V, (3, 7)); src[0, 5:] = 0
    tgt = torch.randint(5, V, (3, 6)); tgt[1, 4:] = 0
    mem1, ok = m.encode(src)
    mem2, _ = m.encode(src, need_weights=True)
    assert torch.allclose(mem1, mem2, atol=1e-5)
    l1 = m.decode(tgt, mem1, ok)
    l2 = m.decode(tgt, mem2, ok, need_weights=True)
    assert torch.allclose(l1, l2, atol=1e-4)


def t_causality():
    torch.manual_seed(0)
    m = _tiny_s2s().eval()
    src = torch.randint(5, V, (2, 6))
    tgt = torch.randint(5, V, (2, 8))
    a = m(src, tgt)
    tgt2 = tgt.clone(); tgt2[:, 5:] = torch.randint(5, V, (2, 3))
    b = m(src, tgt2)
    assert torch.allclose(a[:, :5], b[:, :5], atol=1e-5), "decoder nhìn trộm tương lai!"
    g = _tiny_gpt().eval()
    x = torch.randint(5, V, (2, 10))
    y = x.clone(); y[:, 6:] = torch.randint(5, V, (2, 4))
    assert torch.allclose(g(x)[:, :6], g(y)[:, :6], atol=1e-5), "GPT nhìn trộm tương lai!"


def t_padding_invariance():
    torch.manual_seed(0)
    m = _tiny_s2s().eval()
    src = torch.randint(5, V, (1, 5)); tgt = torch.randint(5, V, (1, 4))
    src_p = torch.cat([src, torch.zeros(1, 3, dtype=torch.long)], 1)
    assert torch.allclose(m(src, tgt), m(src_p, tgt), atol=1e-5), "đệm nguồn làm thay đổi kết quả"
    g = _tiny_gpt().eval()
    x = torch.randint(5, V, (1, 6))
    xp = torch.cat([torch.zeros(1, 3, dtype=torch.long), x], 1)
    mp = xp != 0
    out = g(xp, mp)
    assert not torch.isnan(out).any(), "NaN khi đệm trái"
    assert torch.allclose(g(x)[:, -1], out[:, -1], atol=1e-5), "đệm trái làm thay đổi kết quả GPT"


def t_beam1_equals_greedy():
    torch.manual_seed(0)
    m = _tiny_s2s().eval()
    src = torch.randint(5, V, (4, 6)); src[:, -1] = D.EOS_ID
    mem, ok = m.encode(src)
    bos = torch.full((4, 1), D.BOS_ID)

    def nl(gen):
        lg = m.decode(torch.cat([bos[:gen.size(0)], gen], 1), mem[:gen.size(0)], ok[:gen.size(0)])
        return torch.log_softmax(lg[:, -1], -1)
    g = DEC._greedy_core(nl, 4, 15, src.device)
    b = DEC._beam_core(nl, 4, 1, 15, src.device)
    assert g == b, f"{g} != {b}"


def t_beam_finds_better_or_equal():
    torch.manual_seed(1)
    m = _tiny_s2s().eval()
    src = torch.randint(5, V, (3, 6)); src[:, -1] = D.EOS_ID
    g = DEC.seq2seq_generate_batch(m, src, beam_size=1)
    b = DEC.seq2seq_generate_batch(m, src, beam_size=5)
    assert len(b) == 3 and all(isinstance(x, list) for x in b)

    def score(i, seq):
        tgt_in = torch.tensor([[D.BOS_ID] + seq])
        tgt_out = torch.tensor([seq + [D.EOS_ID]])
        lp = torch.log_softmax(m(src[i:i + 1], tgt_in), -1)
        return lp.gather(-1, tgt_out.unsqueeze(-1)).sum().item() / len(tgt_out[0])
    for i in range(3):
        assert score(i, b[i]) >= score(i, g[i]) - 1e-4, "beam cho điểm kém hơn greedy"


def t_batch_equals_single():
    torch.manual_seed(0)
    g = _tiny_gpt().eval()
    srcs = [[5, 6, 7, 8, 9], [10, 11], [12, 13, 14]]
    batch = DEC.gpt_generate_batch(g, srcs, SEP, beam_size=1, device=torch.device("cpu"))
    single = [DEC.gpt_generate_batch(g, [s], SEP, beam_size=1, device=torch.device("cpu"))[0] for s in srcs]
    assert batch == single, "GPT: dịch theo batch khác dịch từng câu (lỗi đệm trái / vị trí)"
    bb = DEC.gpt_generate_batch(g, srcs, SEP, beam_size=3, device=torch.device("cpu"))
    bs = [DEC.gpt_generate_batch(g, [s], SEP, beam_size=3, device=torch.device("cpu"))[0] for s in srcs]
    assert bb == bs, "GPT beam: batch khác từng câu"
    m = _tiny_s2s().eval()
    s2 = [(i, s, None) for i, s in enumerate(srcs)]
    b = DEC.seq2seq_generate_batch(m, D.seq2seq_collate(s2)["src"], beam_size=3)
    one = [DEC.seq2seq_generate_batch(m, D.seq2seq_collate([x])["src"], beam_size=3)[0] for x in s2]
    assert b == one, "Seq2seq beam: batch khác từng câu"


def t_label_smoothing():
    torch.manual_seed(0)
    logits = torch.randn(2, 5, V)
    tgt = torch.randint(0, V, (2, 5)); tgt[0, 3:] = -100
    a, _, n = MD.label_smoothed_ce(logits, tgt, 0.1)
    b = F.cross_entropy(logits.reshape(-1, V), tgt.reshape(-1), ignore_index=-100, label_smoothing=0.1)
    assert abs(a.item() - b.item()) < 1e-5 and n == 8, (a.item(), b.item(), n)


def t_bleu_matches_sacrebleu():
    rng = random.Random(0)
    words = "tôi bạn là học sinh giáo viên yêu Việt Nam . , một hai ba".split()
    refs = [" ".join(rng.choice(words) for _ in range(rng.randint(3, 15))) for _ in range(200)]
    hyps = [" ".join(w if rng.random() < .7 else rng.choice(words) for w in r.split()) for r in refs]
    ours = M.bleu_from_stats(M.bleu_stats(hyps, refs).sum(0))
    ref = M.quick_bleu(hyps, refs)
    assert abs(ours - ref) < 0.01, f"{ours} vs {ref}"


def _toy_pairs(n, rng):
    # Bài toán đồ chơi: "dịch" = đảo ngược chuỗi và cộng 1 vào mỗi token
    pairs = []
    for _ in range(n):
        s = [rng.randint(5, V - 2) for _ in range(rng.randint(2, 8))]
        pairs.append((s, [x + 1 for x in reversed(s)]))
    return pairs


def _train_toy(kind, steps, ckpt_dir, max_steps):
    rng = random.Random(0)
    torch.manual_seed(0)
    pairs = _toy_pairs(3000, rng)
    src, tgt = [p[0] for p in pairs], [p[1] for p in pairs]
    ds = D.PairDataset(src, tgt)
    if kind == "s2s":
        model = MD.TransformerSeq2Seq(MD.Seq2SeqConfig(vocab_size=V, d_model=64, n_heads=4, d_ff=128, enc_layers=2,
                                                      dec_layers=2, dropout=0.0, attn_dropout=0.0, max_len=64))
        collate = D.seq2seq_collate
        lengths = [max(len(s), len(t)) + 1 for s, t in pairs]

        def loss_fn(m, b, cfg):
            return MD.label_smoothed_ce(m(b["src"], b["tgt_in"]), b["tgt_out"], cfg.label_smoothing, ignore_index=0)
    else:
        model = MD.GPTDecoder(MD.GPTConfig(vocab_size=V, d_model=64, n_heads=4, d_ff=128, n_layers=3, dropout=0.0,
                                           attn_dropout=0.0, max_len=64))
        collate = partial(D.gpt_collate, sep=SEP)
        lengths = [len(s) + len(t) + 3 for s, t in pairs]

        def loss_fn(m, b, cfg):
            return MD.label_smoothed_ce(m(b["input_ids"], b["attn_mask"]), b["labels"], cfg.label_smoothing)
    test = _toy_pairs(50, random.Random(99))

    def eval_fn(m):
        if kind == "s2s":
            out = DEC.seq2seq_generate_batch(m, D.seq2seq_collate([(i, s, None) for i, (s, _) in enumerate(test)])["src"])
        else:
            out = DEC.gpt_generate_batch(m, [s for s, _ in test], SEP, device=torch.device("cpu"))
        acc = float(np.mean([o == t for o, (_, t) in zip(out, test)]))
        return {"score": 100 * acc}
    cfg = TrainConfig(epochs=100, max_tokens=600, lr=3e-3, warmup_steps=50, eval_every_steps=steps,
                      save_every_steps=steps, log_every_steps=steps, max_steps=max_steps, patience=100,
                      label_smoothing=0.0, amp="fp32")
    tr = Trainer(model, ds, collate, lengths, cfg, ckpt_dir, loss_fn, eval_fn, device=torch.device("cpu"),
                 run_name=f"toy-{kind}")
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        tr.train()
    return tr


def t_train_resume_and_learn():
    # (loại mô hình, tổng số bước, ngưỡng độ chính xác tối thiểu)
    # Mô hình chưa học: độ chính xác (khớp cả chuỗi) ≈ 0%.  Ngưỡng 50% đủ chứng minh mô hình học đúng,
    # đồng thời chừa biên cho sai khác số học giữa các máy/phiên bản PyTorch (đã thấy 76%–92% tuỳ máy).
    for kind, total, thr in (("s2s", 600, 50), ("gpt", 1200, 50)):
        with tempfile.TemporaryDirectory() as d:
            # Chạy 150 bước -> giả lập "mất kết nối" -> chạy lại, phải tiếp tục từ bước 150
            tr1 = _train_toy(kind, 150, d, 150)
            assert tr1.state["global_step"] == 150
            tr2 = _train_toy(kind, 150, d, total)
            assert tr2.state["global_step"] == total, tr2.state["global_step"]
            best = tr2.state["best_score"]
            assert best >= thr, f"{kind}: mô hình không học được bài toán đồ chơi (acc={best:.0f}%)"
            losses = [h["train_loss"] for h in tr2.history if "train_loss" in h]
            assert losses[-1] < 0.5 * losses[0], f"{kind}: loss giảm quá ít ({losses[0]:.2f} -> {losses[-1]:.2f})"
            assert (Path(d) / "best.pt").exists() and (Path(d) / "history.csv").exists()


TESTS = [
    ("Chuẩn hoá Unicode tiếng Việt", t_normalize),
    ("Làm sạch dòng (HTML + detokenize)", t_clean_line),
    ("Batch theo số token, xác định theo epoch", t_sampler),
    ("Ghép batch Seq2Seq / GPT (mặt nạ nhãn)", t_collate),
    ("Attention: SDPA == cài đặt thủ công", t_attention_sdpa_vs_manual),
    ("Mặt nạ nhân quả (không nhìn tương lai)", t_causality),
    ("Bất biến với padding (kể cả đệm trái GPT)", t_padding_invariance),
    ("Beam search K=1 == greedy", t_beam1_equals_greedy),
    ("Beam search không kém greedy", t_beam_finds_better_or_equal),
    ("Dịch theo batch == dịch từng câu", t_batch_equals_single),
    ("Label smoothing CE == PyTorch", t_label_smoothing),
    ("BLEU tự tính == sacreBLEU", t_bleu_matches_sacrebleu),
    ("Huấn luyện + resume + học được bài toán đồ chơi", t_train_resume_and_learn),
]


def run_all(skip_slow=False):
    ok = True
    for name, fn in TESTS:
        if skip_slow and fn is t_train_resume_and_learn:
            continue
        try:
            fn()
            print(f"✅ {name}")
        except Exception as e:  # noqa: BLE001
            ok = False
            print(f"❌ {name}: {e}")
            traceback.print_exc()
    print("\n" + ("🎉 TẤT CẢ KIỂM THỬ ĐỀU QUA" if ok else "⛔ CÓ KIỂM THỬ THẤT BẠI — KHÔNG chạy huấn luyện cho tới khi sửa xong"))
    return ok


if __name__ == "__main__":
    import sys
    sys.exit(0 if run_all() else 1)

Overwriting envi_lib/selftest.py


In [14]:
#@title 📦 Cài các gói còn thiếu (không đụng tới torch / transformers có sẵn của Colab)
%pip install -q sacrebleu sacremoses sentencepiece

In [15]:
#@title 🔍 Kiểm tra môi trường
import importlib, torch, transformers, numpy, sacrebleu, sentencepiece
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
importlib.invalidate_caches()
import envi_lib
from envi_lib.common import gpu_info, disk_free_gb
print("envi_lib      :", envi_lib.LIB_VERSION)
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("numpy         :", numpy.__version__)
print("sacrebleu     :", sacrebleu.__version__)
print("GPU           :", gpu_info())
print("Drive còn trống (GB):", disk_free_gb(PROJECT_DIR))
if not torch.cuda.is_available():
    print("⚠️  Chưa bật GPU: Runtime → Change runtime type → chọn GPU (A100/L4/T4). Notebook 00/01 không cần GPU.")

envi_lib      : 1.0.0
torch         : 2.11.0+cu130
transformers  : 5.20.1
numpy         : 2.1.3
sacrebleu     : 2.6.0
GPU           : {'cuda': True, 'torch': '2.11.0+cu130', 'name': 'NVIDIA L4', 'total_gb': 22.0, 'bf16': True}
Drive còn trống (GB): 183.6


In [16]:
#@title 🧪 Chạy bộ kiểm thử tự động (≈1–2 phút)
from envi_lib import selftest
ok = selftest.run_all()
assert ok, "Có kiểm thử thất bại — dừng lại, không chạy các notebook sau."

✅ Chuẩn hoá Unicode tiếng Việt
✅ Làm sạch dòng (HTML + detokenize)
✅ Batch theo số token, xác định theo epoch
✅ Ghép batch Seq2Seq / GPT (mặt nạ nhãn)
✅ Attention: SDPA == cài đặt thủ công
✅ Mặt nạ nhân quả (không nhìn tương lai)
✅ Bất biến với padding (kể cả đệm trái GPT)
✅ Beam search K=1 == greedy
✅ Beam search không kém greedy
✅ Dịch theo batch == dịch từng câu
✅ Label smoothing CE == PyTorch
✅ BLEU tự tính == sacreBLEU


toy-s2s epoch 1/100:   0%|          | 0/30 [00:00<?, ?it/s]

toy-s2s epoch 2/100:   0%|          | 0/30 [00:00<?, ?it/s]

toy-s2s epoch 3/100:   0%|          | 0/30 [00:00<?, ?it/s]

toy-s2s epoch 4/100:   0%|          | 0/30 [00:00<?, ?it/s]

toy-s2s epoch 5/100:   0%|          | 0/30 [00:00<?, ?it/s]

... (33 outputs left out)

toy-gpt epoch 17/100:   0%|          | 0/66 [00:00<?, ?it/s]

toy-gpt epoch 18/100:   0%|          | 0/66 [00:00<?, ?it/s]

toy-gpt epoch 19/100:   0%|          | 0/66 [00:00<?, ?it/s]

✅ Huấn luyện + resume + học được bài toán đồ chơi

🎉 TẤT CẢ KIỂM THỬ ĐỀU QUA


## ✅ Xong
Tiếp theo: mở **`01_du_lieu_tokenizer.ipynb`**.

*Ghi chú:* nếu sau này nhận bản `envi_lib` mới, chỉ cần chạy lại notebook này (dữ liệu và checkpoint không bị ảnh hưởng).